# Family Guys | Drug review rating classification

**GitHub repository:** [Browse the project code](https://github.com/Srappit/dsai4205-family-guys-drug-reviews).

Run Jupyter from the repository root. Run one section at a time and restart the kernel between sections. The weighted baseline and advanced models share the data exported in Section 2. The original ten-class baseline and the historical four-class comparison use the earlier archived split. Refer to the README for the inputs needed for these sections.

The historical four-class baseline reported 59.48% accuracy and 43.86% macro-F1 on validation. The weighted bigram baseline reported 58.50% accuracy and 47.33% macro-F1 on test. Each result retains its original evaluation split.


## Section 1. Data cleaning

Inspect raw reviews, remove duplicate review-rating pairs, reconcile conflicting ratings, and normalize and tokenize the review text.

Run from the repository root.

[Standalone notebook](data/01_clean_data.ipynb)

In [ ]:

import os
import sys

# Set environment variables
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# Create a SparkSession
spark = (
    SparkSession.builder
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)
# Initialize
ss = spark

sc = ss.sparkContext

ss.sparkContext.setLogLevel("ERROR")
# from pyspark.ml.feature import CountVectorizer, IDF, Tokenizer


### Full Steps of Data Cleaning
- Delete duplicate data
- Resolve scoring conflicts
- Remove missing values
- Handle invalid values (non-existent)
- Data standardization
- Stemming
- String tokenization
- Tokenize single‑quoted concatenated terms
- Remove stop words
- split dataset


In [ ]:
# Convert to a DataFrame
data = ss.read.csv(path='drugsComTrain_raw.csv',
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True,
                        quote='"',        # Explicitly specify double quotes as the text delimiter
                        escape='"',       # Escape internal quotes by doubling them
                        multiLine=True    # Allow a field to span multiple lines
                        )


data.select('usefulCount').sort('usefulCount') # Data looks valid; minimum value = 0
data.show(20, truncate=100)
 

In [ ]:
# Keep only review and rating 

df_data = data.select('review','rating').toDF('review','rating')
df_data.count()



In [ ]:
# # Remove duplicates
df_data = df_data.distinct()
# df_data.count()
df_data.show(20, truncate=100)

In [ ]:
# Label conflicts: same review with different ratings
# Count reviews with more than one rating
from pyspark.sql.functions import col, trim
conflict_reviews = (
    df_data.groupBy("review")
    .agg(
        f.countDistinct("rating").alias("different_ratings")
    )
    .filter(col("different_ratings") > 1)
)

conflict_records = df_data.join(
    conflict_reviews.select("review"),
    on="review",
    how="inner"
)

print("涉及评分冲突的行数：", conflict_records.count())

conflict_reviews.sort("different_ratings", ascending=False).show(20)
# The maximum number of duplicates is 6, and the minimum is 2




In [ ]:
from pyspark.sql import functions as f
# For conflicting reviews, merge into one row and round the mean rating to an integer
conflict_rating = (
    df_data.join(conflict_reviews.select("review"), on="review", how="inner")
    .groupBy("review")
    .agg(
        f.round(f.avg("rating")).cast("int").alias("rating")
    )
)
# conflict_rating.show(5)
# conflict_rating.count()

# 3. Select normal data without conflicts
normal_data = df_data.join(
    conflict_reviews.select("review"),
    on="review",
    how="leftanti"   # leftanti: Keep only left-side rows with no match (non-conflicting rows)
)
# 4. Combine normal data with the merged conflicting rows
df_clean = normal_data.unionByName(conflict_rating)
# df_clean.count()


In [ ]:
# Handle missing values
df_clean = df_clean.filter(
    (col("review").isNotNull()) & (trim(col("review")) != "") &
    (col("rating").isNotNull()) & (trim(col("rating")) != "")
)
df_clean.count()

In [ ]:
# Check whether the data is valid
df_clean = df_clean.filter(
    col("rating").between(1, 10))

df_clean.count() 

In [ ]:
# Normalize data
import html
import re
import unicodedata
from pyspark.sql.types import StringType

def normalize_review(text):
    if text is None:
        return None
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    # Remove common HTML tags
    text = re.sub(r"</?[A-Za-z][^>]*>", " ", text)
    # Normalize curly quotes
    text = text.replace("’", "'").replace("‘", "'")
    # Collapse spaces, newlines, and tabs
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Register the UDF on a separate line
normalize_review = f.udf(normalize_review, returnType=StringType())

df_clean = df_clean.withColumn(
    "review",
    normalize_review(col("review"))
)

# Lowercase; keep letters, numbers, whitespace, and ASCII apostrophes
# Keep apostrophes to avoid breaking contractions such as "don't"
df_norm = df_clean.withColumn(
    "review",
    trim(
        f.regexp_replace(
            f.regexp_replace(
                f.lower(col("review")),
                r"[^\p{L}\p{N}\s']",
                " "
            ),
            r"\s+",
            " "
        )
    )
)
df_norm.show(20, truncate=100)

In [ ]:
# Stemming
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stemming(data_str):
    # input is string
    cleaned_str = ''
    text = data_str.split()

    for word in text:
        cleaned_str+= ps.stem(word) + ' '

    return cleaned_str

#define udf
sent_to_stem_words_udf = f.udf(lambda x: str(stemming(x)), StringType())


df_stem = df_norm.select('review', 'rating', sent_to_stem_words_udf(df_norm.review).alias('stemmed_review'))
df_stem.show(10, truncate=100)

In [ ]:
# Tokenization
from pyspark.ml.feature import  Tokenizer
tokenizer = Tokenizer(inputCol="stemmed_review", outputCol="words") # Tokenizer splits each document into words on whitespace
wordsData = tokenizer.transform(df_stem)
wordsData.show(20, truncate=100)

In [ ]:
# Inspect words containing apostrophes

result = (
    wordsData
    .select(
        "review",
        "rating",
        f.explode("words").alias("token")
    )
    .filter(col("token").contains("'"))
)

# View all unique tokens containing apostrophes
result.select("token").distinct().show(10, truncate=False)



In [ ]:
from pyspark.sql.types import ArrayType
# Expand contractions containing apostrophes
SPECIAL_FORMS = {
    "can't": ["can", "not"],
    "cannot": ["can", "not"],
    "won't": ["will", "not"],
    "shan't": ["shall", "not"],
    # The auxiliary in "ain't" depends on context; retain only the negation
    "ain't": ["not"]
}

# Expansion function
def normalize_tokens(tokens):
    if tokens is None:
        return []

    output = []

    for token in tokens:
        if not token:
            continue

        word = token.lower().replace("’", "'").replace("‘", "'")

        # Remove surrounding quotes, e.g. 'dopey' → dopey
        word = word.strip("'\"")

        if not word:
            continue

        # Special negative forms
        if word in SPECIAL_FORMS:
            output.extend(SPECIAL_FORMS[word])

        # don't → do not；wasn't → was not
        elif word.endswith("n't"):
            base = word[:-3]
            if base:
                output.append(base)
            output.append("not")

        # i'm → i am
        elif word.endswith("'m"):
            output.extend([word[:-2], "am"])

        # you're → you are
        elif word.endswith("'re"):
            output.extend([word[:-3], "are"])

        # i've → i have
        elif word.endswith("'ve"):
            output.extend([word[:-3], "have"])

        # you'll → you will
        elif word.endswith("'ll"):
            output.extend([word[:-3], "will"])

        # "'d" may mean "had" or "would"; retain the stem for now
        elif word.endswith("'d"):
            output.append(word[:-2])

        # "'s" may be possessive, "is", or "has"; retain the stem for now
        # tourette's → tourette; it's → it
        elif word.endswith("'s"):
            output.append(word[:-2])

        else:
            # Do not remove other words containing apostrophes
            output.append(word)

    return [word for word in output if word]

# This returns array<string>
normalize_tokens = f.udf(normalize_tokens, returnType=ArrayType(StringType()))

wordsData = wordsData.withColumn(
    "normalized_words",
    normalize_tokens(col("words"))
)

wordsData.select(
    "words", "normalized_words"
).show(20, truncate=100)


In [ ]:
# Stop-word handling
from pyspark.ml.feature import StopWordsRemover

# Negation words to preserve
protected_words = {
    "not", "no", "nor", "never", "neither", "without"
}

default_stopwords = StopWordsRemover.loadDefaultStopWords("english")

custom_stopwords = [
    word for word in default_stopwords
    if word.lower() not in protected_words
]

remover = StopWordsRemover(
    inputCol="normalized_words",
    outputCol="filtered_words",
    stopWords=custom_stopwords,
    caseSensitive=False
)

cleanData = remover.transform(wordsData)

cleanData.select(
    "normalized_words",
    "filtered_words",
).show(20, truncate=100)



In [ ]:
# 80% training set, 20% test set
train_data, test_data = cleanData.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Cache for convenient reuse later
train_data = train_data.cache()
test_data = test_data.cache()

print("tarin set number:", train_data.count())
print("test set number:", test_data.count())


## Section 2. Prepare the shared model splits

Run this section before the weighted baseline and advanced models. It exports train, validation, and test once, along with their manifest.

Run from the repository root.

[Standalone notebook](data/02_prepare_model_splits.ipynb)

# 数据准备：64% 训练 / 16% 验证 / 20% 测试
保留最新 clean_data.ipynb 的清洗顺序，修复 stemming 丢失 rating 的问题。
先按原流程划分 80% development / 20% test，再将 development 按 80% / 20% 划为 train / validation。
Spark randomSplit 的比例是期望比例，实际行数可能略有偏差。所有组合必须共用一次导出的数据。
设置下面两个路径；不要覆盖上一轮数据。测试集不参与 embedding 训练、选轮次或调参。


In [ ]:
from pathlib import Path
RAW_CSV = Path.cwd() / 'drugsComTrain_raw.csv'
OUTPUT_DIR = RAW_CSV.parent / 'data_64_16_20'
assert RAW_CSV.is_file(), RAW_CSV


In [ ]:

import os
import sys

# 设置环境变量
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# 创建 SparkSession
spark = (
    SparkSession.builder
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false")
    .config("spark.python.worker.reuse", "false")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)
#初始化
ss = spark

sc = ss.sparkContext

ss.sparkContext.setLogLevel("ERROR")
# from pyspark.ml.feature import CountVectorizer, IDF, Tokenizer


### 数据清洗全步骤
- 删除缺失值
- 无效值处理（不存在）
- 删除重复数据
- 评分冲突处理
- 数据标准化
- 词干提取
- 字符串分词
- 单引号连接词分词
- 删除停用词

In [ ]:
# 转化为dataframe
data = ss.read.csv(path=str(RAW_CSV),
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True,
                        quote='"',        # 明确指定包裹文本的引号是双引号
                        escape='"',       #内部引号用 "" 转义
                        multiLine=True    #允许一个字段跨多行
                        )


data.select('usefulCount').sort('usefulCount') #数据正常 最小值=0
data.show(20, truncate=100)
 

In [ ]:
#只保留review rating 
df_data = data.select('review','rating').toDF('review','rating')

from pyspark.sql.functions import col, count, trim

# 按 review、rating 分组，统计每组条数
dup_group = df_data.groupBy("review", "rating")\
    .agg(count("*").alias("cnt"))\
    .filter(col("cnt") >= 2)  # 保留重复的组合

dup_group.show() 
# 有多少组重复的(review,rating)组合
dup_group.count() #48652

# 去重处理
df_data = df_data.distinct()
df_data.count()


In [ ]:
from pyspark.sql import functions as f

# 标签冲突：相同review，rating不同
#1. 统计rating不止一个的review
conflict_reviews = (
    df_data.groupBy("review")
    .agg(
        f.countDistinct("rating").alias("different_ratings")
    )
    .filter(col("different_ratings") > 1)
)

conflict_records = df_data.join(
    conflict_reviews.select("review"),
    on="review",
    how="inner"
)

print("涉及评分冲突的行数：", conflict_records.count())

conflict_reviews.sort("different_ratings", ascending=False).show(20)
# 最多重复数据有6条，最少两条




In [ ]:
# 2.对冲突review：合并成一行，rating取均值四舍五入，转化为int
conflict_rating = (
    df_data.join(conflict_reviews.select("review"), on="review", how="inner")
    .groupBy("review")
    .agg(
        f.round(f.avg("rating")).cast("int").alias("rating")
    )
)
conflict_rating.show(5)
conflict_rating.count()

# 3. 取出【没有冲突的正常数据】
normal_data = df_data.join(
    conflict_reviews.select("review"),
    on="review",
    how="leftanti"   # leftanti：只保留左边匹配不到的行（非冲突）
)
# 4. 拼接：正常数据 + 合并后的冲突单行数据
df_clean = normal_data.unionByName(conflict_rating)
df_clean.count()


In [ ]:
#缺失值处理
df_clean = df_clean.filter(
    (col("review").isNotNull()) & (trim(col("review")) != "") &
    (col("rating").isNotNull()) & (trim(col("rating")) != "")
)
df_clean.count()

In [ ]:
# 检查数据是否有效
df_clean = df_clean.filter(
    col("rating").between(1, 10))

df_clean.count() 

In [ ]:
# 标准化处理
import html
import re
import unicodedata
from pyspark.sql.types import StringType

def normalize_review(text):
    if text is None:
        return None
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    # 删除常见 HTML 标签
    text = re.sub(r"</?[A-Za-z][^>]*>", " ", text)
    # 统一弯引号
    text = text.replace("’", "'").replace("‘", "'")
    # 合并空格、换行和制表符
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 单独一行注册UDF
normalize_review = f.udf(normalize_review, returnType=StringType())

df_clean = df_clean.withColumn(
    "review",
    normalize_review(col("review"))
)

# 小写；保留字母、数字、空白和英文单引号
# 保留单引号，避免把 don't 直接破坏掉
df_norm = df_clean.withColumn(
    "review",
    trim(
        f.regexp_replace(
            f.regexp_replace(
                f.lower(col("review")),
                r"[^\p{L}\p{N}\s']",
                " "
            ),
            r"\s+",
            " "
        )
    )
)
df_norm.show(20, truncate=100)

In [ ]:
from nltk.stem import PorterStemmer

# 词干提取需要使用此包
ps = PorterStemmer()

def stemming(data_str):
    # 期望输入为字符串
    cleaned_str = ''
    text = data_str.split()

    for word in text:
        cleaned_str+= ps.stem(word) + ' '

    return cleaned_str

#定义 UDF 函数
sent_to_stem_words_udf = f.udf(lambda x: str(stemming(x)), StringType())


df_stem = df_norm.select('review', 'rating', sent_to_stem_words_udf(df_norm.review).alias('stemmed_review'))
df_stem.show(10, truncate=100)

In [ ]:
# 分词处理
from pyspark.ml.feature import  Tokenizer
tokenizer = Tokenizer(inputCol="stemmed_review", outputCol="words") # Tokenizer 按空格将每篇文档拆分为单词
wordsData = tokenizer.transform(df_stem)
wordsData.show(10, truncate=50)

In [ ]:
#查看含义单引号的word

result = (
    wordsData
    .select(
        "review",
        "rating",
        f.explode("words").alias("token")
    )
    .filter(col("token").contains("'"))
)

# 查看所有包含单引号的唯一 token
result.select("token").distinct().show(10, truncate=False)



In [ ]:
from pyspark.sql.types import ArrayType
#展开单引号连接词
SPECIAL_FORMS = {
    "can't": ["can", "not"],
    "cannot": ["can", "not"],
    "won't": ["will", "not"],
    "shan't": ["shall", "not"],
    # ain't 的具体助动词取决于语境，这里只保留否定信息
    "ain't": ["not"]
}

#展开函数
def normalize_tokens(tokens):
    if tokens is None:
        return []

    output = []

    for token in tokens:
        if not token:
            continue

        word = token.lower().replace("’", "'").replace("‘", "'")

        # 去掉包裹词语的引号，例如 'dopey' → dopey
        word = word.strip("'\"")

        if not word:
            continue

        # 特殊否定形式
        if word in SPECIAL_FORMS:
            output.extend(SPECIAL_FORMS[word])

        # don't → do not；wasn't → was not
        elif word.endswith("n't"):
            base = word[:-3]
            if base:
                output.append(base)
            output.append("not")

        # i'm → i am
        elif word.endswith("'m"):
            output.extend([word[:-2], "am"])

        # you're → you are
        elif word.endswith("'re"):
            output.extend([word[:-3], "are"])

        # i've → i have
        elif word.endswith("'ve"):
            output.extend([word[:-3], "have"])

        # you'll → you will
        elif word.endswith("'ll"):
            output.extend([word[:-3], "will"])

        # 'd 可能是 had 或 would，先保留主体
        elif word.endswith("'d"):
            output.append(word[:-2])

        # 's 可能是所有格、is 或 has，先保留主体
        # tourette's → tourette；it's → it
        elif word.endswith("'s"):
            output.append(word[:-2])

        else:
            # 其他带单引号的词不直接删除
            output.append(word)

    return [word for word in output if word]

#这里返回array<string>
normalize_tokens = f.udf(normalize_tokens, returnType=ArrayType(StringType()))

wordsData = wordsData.withColumn(
    "normalized_words",
    normalize_tokens(col("words"))
)

wordsData.select(
    "words", "normalized_words"
).show(20, truncate=150)


In [ ]:
#停用词处理
from pyspark.ml.feature import StopWordsRemover

# 需要保留的否定词
protected_words = {
    "not", "no", "nor", "never", "neither", "without"
}

default_stopwords = StopWordsRemover.loadDefaultStopWords("english")

custom_stopwords = [
    word for word in default_stopwords
    if word.lower() not in protected_words
]

remover = StopWordsRemover(
    inputCol="normalized_words",
    outputCol="filtered_words",
    stopWords=custom_stopwords,
    caseSensitive=False
)

cleanData = remover.transform(wordsData)

cleanData.select(
    "filtered_words",
    "rating"
).show(20, truncate=150)



In [ ]:
# 80% 训练集，20% 测试集
train_data, test_data = cleanData.randomSplit(
    [0.8, 0.2],
    seed=42
)

# 缓存，方便后续多次使用
train_data = train_data.cache()
test_data = test_data.cache()

print("训练集记录数：", train_data.count())
print("测试集记录数：", test_data.count())


In [ ]:
# Export all three partitions once; model notebooks never split data again.
import json, hashlib
from collections import Counter

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if (OUTPUT_DIR / 'manifest.json').exists():
    raise FileExistsError('Inputs already exported. Use a NEW OUTPUT_DIR to avoid mixing experiments.')
development_data = train_data
train_data, validation_data = development_data.randomSplit([0.8, 0.2], seed=43)
train_data.cache(); validation_data.cache()
split_info, text_groups = {}, {}
for split_name, frame in [('train', train_data), ('validation', validation_data), ('test', test_data)]:
    target = OUTPUT_DIR / (split_name + '.jsonl')
    temporary = target.with_suffix('.jsonl.tmp')
    counts, groups, n, empty = Counter(), set(), 0, 0
    with temporary.open('w', encoding='utf-8', newline='\n') as stream:
        for row in frame.select('review','rating','filtered_words').toLocalIterator():
            words, rating = list(row.filtered_words), int(row.rating)
            assert 1 <= rating <= 10 and all(isinstance(w,str) for w in words)
            group = hashlib.sha256(json.dumps(words,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
            record = dict(row_id=split_name+'_'+str(n), review=row.review,
                          rating=rating, filtered_words=words, text_group_id=group)
            stream.write(json.dumps(record,ensure_ascii=False)+'\n')
            counts[rating] += 1; groups.add(group); n += 1; empty += int(not words)
    temporary.replace(target)
    split_info[split_name] = dict(file=target.name,n=n,rating_counts=dict(counts),empty_token_rows=empty,
                                  sha256=hashlib.sha256(target.read_bytes()).hexdigest())
    text_groups[split_name] = groups
overlap = {a+'_'+b:len(text_groups[a] & text_groups[b]) for a,b in [('train','validation'),('train','test'),('validation','test')]}
manifest = dict(pipeline='latest_clean_data_64_16_20_v2',source_notebook_sha256='12ce8f1a29e9f205ebd96fbd5d0265d9a8fc987e8c9c93b5e7f42232e035f3c1',
                raw_csv=str(RAW_CSV.resolve()),raw_sha256=hashlib.sha256(RAW_CSV.read_bytes()).hexdigest(),
                modifications=['Preserve rating in df_stem','Keep original test_data; subdivide original train_data with seed 43','Export fixed partitions'],
                split_method='Spark 80/20 seed42, then development 80/20 seed43; expected 64/16/20',
                independent_test_available=True,split_info=split_info,normalized_text_group_overlap=overlap)
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(manifest,indent=2,ensure_ascii=False))
if any(overlap.values()):
    print('NOTICE: identical filtered text occurs across partitions. Preserved the supplied randomSplit; disclose this in the report. These are not group-isolated splits.')
print('Use this same data directory for every combination. Do not run randomSplit separately in each model notebook.')


## Section 3. Original baseline: ten ratings

Refit unigram TF-IDF and multinomial logistic regression on development data. Evaluate exact ratings and near misses on the original fixed test split.

Run from the repository root.

[Standalone notebook](baselines/01_original_ten_class.ipynb)

# Final baseline: review-only rating prediction, 80/20 refit

在先前约64/16/20划分上，validation已经用于选择 regParam=0.1。本次把原 train 和 validation 合并为 development（约80%），重新拟合词表、IDF和分类器；固定原 test（约20%），输出最终评估。只训练预选的0.1，不用test调整。

原测试集与其他实验已在项目探索中被查看，不能宣称全新盲测。组员清洗规则在clean_data.csv中已经实施：原始 review/rating distinct，同原始review不同rating的不同评分平均并取整，112,326条有效记录。模型预测1–10评分分布，不是情感标签。


In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, time, json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "clean_data.csv").is_file():
    raise FileNotFoundError("Place the archived clean_data.csv in the repository root.")
CLEAN_CSV = PROJECT_DIR / "clean_data.csv"
EXPECTED_CSV_SHA256 = "a74e8d67a9542a068f05823afd5c1fb34208fff1716e652ec6906485f891e911"
REFERENCE_DIR = PROJECT_DIR / "baseline_results" / "rating_teammate_policy_20261106_155015_111148"
SEED = 42
N_CLASSES = 10
CLASS_NAMES = [str(i) for i in range(1, 11)]
CLASS_TICKS = CLASS_NAMES
REG_PARAM = 0.1  # 固定之前在validation选出的参数
VOCAB_SIZE = 30000
MIN_DF = 2.0
MAX_ITER = 150
RUN_DIR = PROJECT_DIR / "baseline_results" / datetime.now().strftime("rating_teammate_refit80_%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
spark = (SparkSession.builder.appName("Rating_teammate_refit80")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable).getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("输出目录：", RUN_DIR)
print("Python:", platform.python_version(), "Spark:", pyspark.__version__)


## 1. 读取已清洗数据并核对来源

In [ ]:
schema = StructType([
    StructField("review", StringType()),StructField("rating",IntegerType()),
    StructField("filtered_words",StringType())])
if not CLEAN_CSV.exists():
    raise FileNotFoundError(CLEAN_CSV)
clean_hash = hashlib.sha256(CLEAN_CSV.read_bytes()).hexdigest()
assert clean_hash == EXPECTED_CSV_SHA256, "clean_data.csv与已核对版本不同，请先检查来源和行数"
cleanData = (spark.read.schema(schema).option("header",True).option("multiLine",True)
    .option("quote",'"').option("escape",'"').option("encoding","UTF-8")
    .option("mode","FAILFAST").csv(str(CLEAN_CSV))
    .withColumn("filtered_words",F.from_json("filtered_words",ArrayType(StringType()))))
valid = F.coalesce(F.col("review").isNotNull() & (F.length("review")>0)
    & F.col("rating").between(1,10) & (F.size("filtered_words")>0), F.lit(False))
assert cleanData.filter(~valid).limit(1).count()==0
assert cleanData.count()==112326
# CSV没有原始uniqueID；构造仅用于审计/导出用的稳定行键，不作模型输入。
cleanData = (cleanData.withColumn("review_original",F.col("review"))
    .withColumn("uniqueID",F.sha2(F.concat_ws("|",F.col("review"),F.col("rating").cast("string")),256))
    .withColumn("text_group_id",F.sha2(F.to_json("filtered_words"),256)))
print("组员规则清洗后的有效行数：",cleanData.count())


## 2. 固定原测试集，合并训练与验证

哈希规则沿用上一版。下面核对测试集中每个（文本组，评分）的计数，考虑清洗后可能相同的多条记录；不使用测试评分选择参数。

In [ ]:
model_data = (cleanData.withColumn("label", F.col("rating").cast("double") - 1.0)
    .withColumn("bucket",F.pmod(F.xxhash64("text_group_id",F.lit(SEED)),F.lit(10000)))
    .withColumn("split",F.when(F.col("bucket") < 8000,"development").otherwise("test"))
    .drop("bucket").cache())
development=model_data.filter(F.col("split")=="development").cache()
test=model_data.filter(F.col("split")=="test").cache()
assert model_data.groupBy("text_group_id").agg(F.countDistinct("split").alias("n")).filter("n>1").count()==0
assert development.count()==89912 and test.count()==22414
for name,df in [("development",development),("test",test)]:
    labels={int(r.label) for r in df.select("label").distinct().collect()}
    assert labels==set(range(10)),f"{name}缺少类别"
manifest=model_data.select("text_group_id","rating","label","split").toPandas()
reference=pd.read_csv(REFERENCE_DIR / "split_manifest.csv")
cols=["text_group_id","rating","label"]
# CSV reads rating as int64; Spark may return int32. Normalize key types before grouping.
def test_record_counts(frame):
    keys=frame.loc[frame.split == "test", cols].copy()
    if keys.isna().any().any():
        raise ValueError("测试集记录键含缺失值")
    keys["text_group_id"]=keys["text_group_id"].astype("string")
    for column in ["rating", "label"]:
        numeric=pd.to_numeric(keys[column], errors="raise")
        if not (numeric == np.floor(numeric)).all():
            raise ValueError(f"{column} 应为整数")
        keys[column]=numeric.astype("int64")
    return keys.groupby(cols,dropna=False).size().sort_index()
old_counts=test_record_counts(reference)
new_counts=test_record_counts(manifest)
pd.testing.assert_series_equal(old_counts,new_counts,check_exact=True)
manifest.to_csv(RUN_DIR/"split_manifest.csv",index=False)
distribution=model_data.groupBy("split","rating").count().orderBy("split","rating").toPandas()
distribution.to_csv(RUN_DIR/"split_distribution.csv",index=False)
display(distribution.pivot(index="rating",columns="split",values="count"))
print("development:",development.count(),"test:",test.count(),"测试集与原实验一致")


## 3. 在完整 development 集合拟合 unigram TF-IDF 和 LR

词表及IDF在 development 拟合，不使用 test 信息。

In [ ]:
feature_model=Pipeline(stages=[CountVectorizer(inputCol="filtered_words",outputCol="term_counts",vocabSize=VOCAB_SIZE,minDF=MIN_DF,binary=False),
    IDF(inputCol="term_counts",outputCol="features")]).fit(development)
development_features=feature_model.transform(development).cache()
test_features=feature_model.transform(test).cache()
start=time.perf_counter()
lr=LogisticRegression(featuresCol="features",labelCol="label",family="multinomial",regParam=REG_PARAM,
    elasticNetParam=0.0,maxIter=MAX_ITER,standardization=True,tol=1e-6)
model=lr.fit(development_features)
iterations=int(model.summary.totalIterations)
print("训练秒数:",round(time.perf_counter()-start,2),"迭代次数:",iterations,"词表大小:",len(feature_model.stages[0].vocabulary))
if iterations>=MAX_ITER: print("注意：达到迭代上限，需检查收敛。")
(RUN_DIR/"feature_metadata.json").write_text(json.dumps({"vocabulary":feature_model.stages[0].vocabulary},ensure_ascii=False),encoding="utf-8")
np.savez_compressed(RUN_DIR/"parameters.npz",idf=feature_model.stages[1].idf.toArray(),coefficients=model.coefficientMatrix.toArray(),intercept=model.interceptVector.toArray())
pd.DataFrame({"iteration":range(len(model.summary.objectiveHistory)),"objective":model.summary.objectiveHistory}).to_csv(RUN_DIR/"objective_history.csv",index=False)


## 4. 评价固定测试集

同时输出十分类 Accuracy、每类 Precision/Recall/F1、整数预测±1/±2命中率以及连续期望评分的 MAE/RMSE。

In [ ]:
def metrics_from_matrix(cm):
    cm = np.asarray(cm, dtype=np.int64)
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    tp = np.diag(cm)
    precision = np.divide(tp, predicted, out=np.zeros(N_CLASSES), where=predicted != 0)
    recall = np.divide(tp, support, out=np.zeros(N_CLASSES), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(N_CLASSES), where=precision + recall != 0)
    n = int(cm.sum())
    if n == 0:
        raise ValueError('评估集合为空')
    weights = support / n
    summary = {'n': n, 'accuracy': float(tp.sum() / n), 'precision_macro': float(precision.mean()), 'recall_macro': float(recall.mean()), 'f1_macro': float(f1.mean()), 'precision_weighted': float(precision @ weights), 'recall_weighted': float(recall @ weights), 'f1_weighted': float(f1 @ weights)}
    per_class = pd.DataFrame({'class_id': np.arange(N_CLASSES), 'class_name': CLASS_NAMES, 'precision': precision, 'recall': recall, 'f1': f1, 'support': support})
    return (summary, per_class)

def rating_decisions(probabilities):
    p = np.asarray(probabilities,dtype=float)
    if p.ndim != 2 or p.shape[1] != 10 or not np.isfinite(p).all():
        raise ValueError("预测概率形状或数值无效")
    if (p < 0).any() or not np.allclose(p.sum(axis=1),1.0,atol=1e-6):
        raise ValueError("评分概率不满足概率分布要求")
    return {
        "map":p.argmax(axis=1)+1,
        "expected":p @ np.arange(1,11,dtype=float),
        "median":(np.cumsum(p,axis=1) >= 0.5).argmax(axis=1)+1
    }

def evaluate_rating_frame(predictions):
    local = predictions.select("uniqueID","text_group_id","rating",vector_to_array("probability").alias("probability")).toPandas()
    if len(local)==0:
        raise ValueError("评估集为空")
    y=local.rating.to_numpy(dtype=int)
    decisions=rating_decisions(np.stack(local.probability.to_numpy()))
    summary={"n":len(local)}
    for decision, values in decisions.items():
        error=np.abs(values-y)
        summary.update({"mae_"+decision:float(error.mean()),
                        "rmse_"+decision:float(np.sqrt(np.mean((values-y)**2))),
                        "within_1_"+decision:float(np.mean(error <= 1.0+1e-10)),
                        "within_2_"+decision:float(np.mean(error <= 2.0+1e-10))})
        if decision != "expected":
            summary["exact_accuracy_"+decision]=float(np.mean(values==y))
        local["predicted_"+decision]=values
    cm=np.zeros((10,10),dtype=np.int64)
    np.add.at(cm,(y-1,decisions["map"]-1),1)
    classification, report=metrics_from_matrix(cm)
    summary.update({"classification_"+k:v for k,v in classification.items() if k!='n'})
    report["predicted_count"]=cm.sum(axis=0)
    local["absolute_error_expected"]=np.abs(local.predicted_expected-y)
    local["probability"]=local.probability.map(lambda x:json.dumps([float(v) for v in x]))
    return summary,report,cm,local


In [ ]:
summary,report,cm,predictions=evaluate_rating_frame(model.transform(test_features))
(RUN_DIR/"final_test_metrics.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
report.to_csv(RUN_DIR/"final_test_classification_report.csv",index=False)
pd.DataFrame(cm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(RUN_DIR/"final_test_map_confusion.csv")
predictions.to_csv(RUN_DIR/"final_test_predictions.csv",index=False)
print("accuracy",round(summary["exact_accuracy_map"],4),"±1:",round(summary["within_1_map"],4),"±2:",round(summary["within_2_map"],4),"expected MAE:",round(summary["mae_expected"],4))
display(pd.DataFrame([summary]))
display(report)
fig,ax=plt.subplots(figsize=(8,7));im=ax.imshow(cm,cmap="Blues")
ax.set(xticks=range(10),yticks=range(10),xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,xlabel="Predicted rating",ylabel="Observed rating",title="Test confusion matrix")
for i in range(10):
    for j in range(10):ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=6,color="white" if cm[i,j]>cm.max()/2 else "black")
fig.colorbar(im,ax=ax);fig.tight_layout();fig.savefig(RUN_DIR/"test_confusion_matrix.png",dpi=150);plt.show();plt.close(fig)
config={"model":"TF-IDF + multinomial logistic regression","data_sha256":clean_hash,"seed":SEED,
    "split":"same prior test; old train+validation merged into development","development_n":89912,"test_n":22414,
    "parameter_selection":"regParam=0.1 selected earlier by validation expected MAE","reg_param":REG_PARAM,
    "vocab_size_limit":VOCAB_SIZE,"actual_vocab_size":len(feature_model.stages[0].vocabulary),"min_df":MIN_DF,"max_iter":MAX_ITER,"iterations":iterations,
    "test_caveat":"Test was previously viewed during exploration; not an untouched blind test",
    "prior_run":str(REFERENCE_DIR),"spark":pyspark.__version__}
(RUN_DIR/"experiment_config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")
print("输出目录:",RUN_DIR)


## Section 4. Improved baseline: four rating bands

Fit bigram TF-IDF and square-root-weighted Spark logistic regression. The archived test output reports 58.50% accuracy and 47.33% macro-F1.

Run from the repository root.

[Standalone notebook](baselines/02_improved_four_class.ipynb)

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, time, json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array

PROJECT_DIR = Path.cwd()
DATA_DIR = Path.cwd() / 'data_64_16_20'

SEED = 42
N_CLASSES = 4
CLASS_NAMES = ['low', 'moderate', 'good', 'excellent']
REG_PARAM = 0.1
VOCAB_SIZE = 30000
MIN_DF = 2.0
MAX_ITER = 50  # <--- 调小迭代次数，防止内存溢出

RUN_DIR = PROJECT_DIR / "baseline_results" / datetime.now().strftime("rating_4class_%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ================= 修改：增加 Spark 内存与超时配置 =================
spark = (SparkSession.builder.appName("Rating_4class")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.driver.memory", "4g")       # 分配更多 Driver 内存
    .config("spark.executor.memory", "4g")     # 分配更多 Executor 内存
    .config("spark.network.timeout", "600s")   # 增加网络超时时间
    .getOrCreate())
# ======================================================================

spark.sparkContext.setLogLevel("ERROR")
print("输出目录：", RUN_DIR)
print("Python:", platform.python_version(), "Spark:", pyspark.__version__)

In [ ]:
# 假设 jsonl 里的字段名为 review, rating, filtered_words
schema = StructType([
    StructField("review", StringType()),
    StructField("rating", IntegerType()),
    StructField("filtered_words", ArrayType(StringType()))
])

def load_jsonl(path, split_name):
    return spark.read.schema(schema).json(str(path)).withColumn("split", F.lit(split_name))

print("正在读取 JSONL 数据...")
df_train = load_jsonl(DATA_DIR / "train.jsonl", "development")
df_val = load_jsonl(DATA_DIR / "validation.jsonl", "development")
df_test = load_jsonl(DATA_DIR / "test.jsonl", "test")

cleanData = df_train.union(df_val).union(df_test)

# 基础有效性校验条件
valid = F.coalesce(
    F.col("review").isNotNull() & (F.length("review") > 0)
    & F.col("rating").between(1, 10) & (F.size("filtered_words") > 0),
    F.lit(False))

print(f"过滤前数据总量: {cleanData.count()}")
cleanData = cleanData.filter(valid)
print(f"过滤后有效数据总量: {cleanData.count()}")

cleanData = (cleanData.withColumn("review_original", F.col("review"))
    .withColumn("uniqueID", F.sha2(F.concat_ws("|", F.col("review"), F.col("rating").cast("string")), 256))
    .withColumn("text_group_id", F.sha2(F.to_json("filtered_words"), 256)))

# ===== 映射为 4 分类标签 =====
model_data = cleanData.withColumn("label", 
    F.when(F.col("rating") <= 3, F.lit(0.0))
    .when(F.col("rating") <= 6, F.lit(1.0))
    .when(F.col("rating") <= 8, F.lit(2.0))
    .otherwise(F.lit(3.0))
).cache()

development = model_data.filter(F.col("split") == "development").cache()
test = model_data.filter(F.col("split") == "test").cache()

dev_count = development.count()
test_count = test.count()
print(f"development 数量: {dev_count}, test 数量: {test_count}")

for name, df in [("development", development), ("test", test)]:
    labels = {int(r.label) for r in df.select("label").distinct().collect()}
    assert labels == set(range(4)), f"{name} 缺少类别: {labels}"

manifest = model_data.select("text_group_id", "rating", "label", "split").toPandas()
manifest.to_csv(RUN_DIR / "split_manifest.csv", index=False)

distribution = model_data.groupBy("split", "rating").count().orderBy("split", "rating").toPandas()
distribution.to_csv(RUN_DIR / "split_distribution.csv", index=False)
display(distribution.pivot(index="rating", columns="split", values="count"))

In [ ]:
import time
from pyspark.sql.functions import col, when, lit
from pyspark.ml.feature import CountVectorizer, IDF, NGram  # 确保导入了 NGram
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

print("开始拟合 TF-IDF (带 2-gram) ...")

# ================= 修改 1：先使用 NGram 生成词组 =================
# 注意：这里只生成 2-gram (相邻的两个词组合)，因为 Spark 不支持在 CountVectorizer 里直接混用 (1,2)
ngram = NGram(n=2, inputCol="filtered_words", outputCol="ngrams")

# 将原始 development 和 test 数据转换，生成包含双词组的 DataFrame
ngram_development = ngram.transform(development)
ngram_test = ngram.transform(test)

# 然后再对生成的双词组进行词频统计和 TF-IDF
feature_model = Pipeline(stages=[
    CountVectorizer(
        inputCol="ngrams",       # <--- 这里必须用 "ngrams"，不是原来的 "filtered_words"
        outputCol="term_counts", 
        vocabSize=VOCAB_SIZE, 
        minDF=MIN_DF, 
        binary=False
    ),
    IDF(inputCol="term_counts", outputCol="features")
]).fit(ngram_development)
# ==================================================================

development_features = feature_model.transform(ngram_development).cache()
test_features = feature_model.transform(ngram_test).cache()

# ================= 修改 2：使用平方根平滑权重 =================
print("计算类别权重 (平方根平滑) ...")
total_count = development_features.count()
counts_dict = {int(row['label']): row['count'] for row in development_features.groupBy("label").count().collect()}

weight_expr = None
for label_val, count in counts_dict.items():
    # 权重公式： (总样本数 / 该类别样本数) 的 0.5 次方
    w = (float(total_count) / count) ** 0.5
    print(f"Label {label_val}: 样本数={count}, 权重={w:.4f}")
    if weight_expr is None:
        weight_expr = when(col("label") == label_val, lit(w))
    else:
        weight_expr = weight_expr.when(col("label") == label_val, lit(w))

dev_weighted = development_features.withColumn("weight", weight_expr).cache()
# =============================================================

print("开始训练逻辑回归 (N-gram + 平滑权重) ...")
start = time.perf_counter()
lr = LogisticRegression(
    featuresCol="features", 
    labelCol="label", 
    weightCol="weight",  # 启用平滑权重
    family="multinomial", 
    regParam=REG_PARAM,
    elasticNetParam=0.0, 
    maxIter=MAX_ITER,
    standardization=True, 
    tol=1e-6
)

model = lr.fit(dev_weighted) 
iterations = int(model.summary.totalIterations) if hasattr(model.summary, 'totalIterations') else MAX_ITER

print(f"训练完成，耗时: {time.perf_counter() - start:.2f} 秒, 迭代次数: {iterations}, 词表大小: {len(feature_model.stages[0].vocabulary)}")

# 保存模型参数
(RUN_DIR / "feature_metadata.json").write_text(json.dumps({"vocabulary": feature_model.stages[0].vocabulary}, ensure_ascii=False), encoding="utf-8")
np.savez_compressed(RUN_DIR / "parameters.npz", idf=feature_model.stages[1].idf.toArray(), coefficients=model.coefficientMatrix.toArray(), intercept=model.interceptVector.toArray())
pd.DataFrame({"iteration": range(len(model.summary.objectiveHistory)), "objective": model.summary.objectiveHistory}).to_csv(RUN_DIR / "objective_history.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import json
import matplotlib.pyplot as plt
from pyspark.ml.functions import vector_to_array

# 各区间映射分数 (用于计算 MAE)
RATING_CENTERS = [2.0, 5.0, 7.5, 9.5] 

def evaluate_4class_model(predictions):
    local = predictions.select("uniqueID", "rating", vector_to_array("probability").alias("probability")).toPandas()
    y_true_rating = local.rating.to_numpy(dtype=int)
    prob_matrix = np.stack(local.probability.to_numpy())
    
    pred_class = prob_matrix.argmax(axis=1)
    assert pred_class.max() <= 3, f"模型输出了 {pred_class.max()} 类别，请检查训练集标签！"
    
    pred_rating = np.array([RATING_CENTERS[c] for c in pred_class])
    
    # 真实类别映射
    y_true_class = np.where(y_true_rating <= 3, 0, 
                   np.where(y_true_rating <= 6, 1, 
                   np.where(y_true_rating <= 8, 2, 3)))
    
    report = classification_report(
        y_true_class, pred_class, 
        labels=range(4), 
        target_names=CLASS_NAMES, 
        output_dict=True, 
        zero_division=0
    )
    
    mae = float(np.mean(np.abs(pred_rating - y_true_rating)))
    rmse = float(np.sqrt(np.mean((pred_rating - y_true_rating) ** 2)))
    within_1 = float(np.mean(np.abs(pred_rating - y_true_rating) <= 1.5))
    within_2 = float(np.mean(np.abs(pred_rating - y_true_rating) <= 2.5))
    
    summary = {
        "accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
        "precision_macro": report["macro avg"]["precision"],
        "recall_macro": report["macro avg"]["recall"],
        "mae_expected": mae,
        "rmse_expected": rmse,
        "within_1": within_1,
        "within_2": within_2
    }
    
    cm = confusion_matrix(y_true_class, pred_class, labels=range(4))
    
    local["pred_class"] = pred_class
    local["pred_rating"] = pred_rating
    local["true_class"] = y_true_class
    local["probability"] = local.probability.map(lambda x: json.dumps([float(v) for v in x]))
    
    return summary, pd.DataFrame(report).transpose(), cm, local

print("正在对测试集进行评估 ...")
predictions = model.transform(test_features)
summary, report_df, cm, local_predictions = evaluate_4class_model(predictions)

(RUN_DIR / "final_test_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
report_df.to_csv(RUN_DIR / "final_test_classification_report.csv")
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RUN_DIR / "final_test_confusion_matrix.csv")
local_predictions.to_csv(RUN_DIR / "final_test_predictions.csv", index=False)

print("\n" + "="*20 + " 最终评估结果 (4分类带权重) " + "="*20)
print(f"accuracy: {summary['accuracy']:.4f}")
print(f"macro_f1: {summary['macro_f1']:.4f}")
print(f"precision_macro: {summary['precision_macro']:.4f}")
print(f"recall_macro: {summary['recall_macro']:.4f}")
print("-" * 50)
print(f"映射预期分数 MAE: {summary['mae_expected']:.4f} | 误差<=1.5 准确率: {summary['within_1']:.4f}")
print("="*50 + "\n")

display(report_df)

# 画图
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
       xlabel="Predicted Category", ylabel="Observed Category", title="4-Class Confusion Matrix (Weighted)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", 
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(RUN_DIR / "test_confusion_matrix.png", dpi=150)
plt.show()
plt.close(fig)

## Section 5. DPCNN with FastText

Fit FastText on training text and train the weighted convolutional classifier. Use validation macro-F1 for checkpoint selection.

Run from the repository root.

[Standalone notebook](models/DPCNN_FastText.ipynb)

# DPCNN＋FastText（平方根加權）
將 notebook 放在含有 data_64_16_20 資料夾的目錄，以該目錄為工作目錄，點選 Run All。預設訓練 10 輪，按驗證 Macro-F1 保存最佳權重。

In [ ]:
# Independent model program: no imports from the old experiment workspace.
import argparse, hashlib, json, random, time, zlib
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix

MODEL_NAME = 'DPCNN'
EMBEDDING_METHOD = 'fasttext'
# ===== 参数区：通常只需修改这里 =====
SEED = 42
DIMENSION = 48          # embedding 维数：六组保持一致
MAX_LENGTH = 96         # 文本最大词数：六组保持一致
WINDOW = 4             # embedding 上下文窗口
BUCKETS = 50000         # FastText 子词哈希桶数
EPOCHS = 10             # 最大训练轮数，按 validation Macro-F1 选最好一轮
EMBEDDING_EPOCHS = 2    # embedding 训练轮数
BATCH_SIZE = 128        # original 64
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01
GRADIENT_CLIP = 5
MIN_COUNT = 2
MAX_VOCAB = 30000
CHANNELS = 128          # DPCNN 卷积通道 / TextRCNN 投影维度
BLOCKS = 4             # 仅 DPCNN：残差块数
KERNEL_SIZE = 3        # 仅 DPCNN：正奇数卷积核
RNN_HIDDEN = 32        # 仅 TextRCNN：每个方向的 LSTM 隐层维数
WEIGHT_ALPHA = 0.5  # 0=普通版；0.5=平方根加权版
# ===== 参数区结束 =====
NAMES = ['low', 'moderate', 'good', 'excellent']
PACKAGE_DIR = Path.cwd()

In [ ]:
def character_ngrams(word):
    bounded = f'<{word}>'
    return [1 + zlib.crc32(bounded[i:i+n].encode()) % BUCKETS
            for n in range(3, 7) for i in range(len(bounded) - n + 1)
            if n != len(bounded)]

def component_table(words):
    rows = [[]] + [[BUCKETS + index, *character_ngrams(word)]
                   for index, word in enumerate(words, start=1)]
    table = torch.zeros((len(rows), max(map(len, rows))), dtype=torch.long)
    for index, row in enumerate(rows):
        table[index, :len(row)] = torch.tensor(row, dtype=torch.long)
    return table

def make_pairs(corpus):
    counts = Counter()
    for sentence in corpus:
        for i, center in enumerate(sentence):
            if center == 0:
                continue
            for j in range(max(0, i - WINDOW), min(len(sentence), i + WINDOW + 1)):
                if i != j and sentence[j] != 0:
                    counts[center, sentence[j]] += 1.0 / abs(i - j)
    pairs = np.asarray(list(counts), dtype=np.int64)
    values = np.fromiter(counts.values(), dtype=np.float32, count=len(counts))
    return torch.from_numpy(pairs), torch.from_numpy(values)

def train_embedding(method, words, pairs, counts, frequency, oov, epochs):
    torch.manual_seed(SEED)
    size = len(words) + 1
    subwords = component_table(words) if method == 'fasttext' else None
    input_size = size + BUCKETS if subwords is not None else size
    source = nn.Embedding(input_size, DIMENSION, padding_idx=0, sparse=True)
    context = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    nn.init.uniform_(source.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    if method == 'glove':
        nn.init.uniform_(context.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
        source_bias, context_bias = nn.Embedding(size, 1, sparse=True), nn.Embedding(size, 1, sparse=True)
        nn.init.zeros_(source_bias.weight)
        nn.init.zeros_(context_bias.weight)
        parameters = [*source.parameters(), *context.parameters(), *source_bias.parameters(), *context_bias.parameters()]
    else:
        nn.init.zeros_(context.weight)
        parameters = [*source.parameters(), *context.parameters()]
    optimizer = torch.optim.SparseAdam(parameters, lr=0.025 if method == 'glove' else 0.01)
    noise = frequency.pow(0.75)

    def centers(ids):
        if subwords is None:
            return source(ids)
        components = subwords[ids]
        return source(components).sum(dim=1) / components.ne(0).sum(dim=1, keepdim=True).clamp_min(1)

    for epoch in range(epochs):
        if method == 'glove':
            order = torch.randperm(len(pairs))
        else:
            # Sample observed pairs by distance-weighted frequency for SGNS.
            order = torch.multinomial(counts, len(pairs), replacement=True)
        loss_sum = 0.0
        for batch in order.split(4096):
            left, right = pairs[batch].T
            vectors = centers(left)
            positive = (vectors * context(right)).sum(dim=1)
            if method == 'glove':
                weights = counts[batch]
                residual = positive + source_bias(left).flatten() + context_bias(right).flatten() - weights.log()
                loss = (((weights / 10).clamp(max=1).pow(0.75)) * residual.square()).mean()
            else:
                negatives = torch.multinomial(noise, len(batch) * 5, replacement=True).reshape(-1, 5)
                negative = torch.einsum('bd,bkd->bk', vectors, context(negatives))
                loss = -(F.logsigmoid(positive) + F.logsigmoid(-negative).sum(dim=1)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f'{method} embedding epoch {epoch + 1}/{epochs} loss={loss_sum / len(order):.4f}', flush=True)

    with torch.no_grad():
        vectors = torch.cat([centers(ids) for ids in torch.arange(size).split(1024)])
        if method == 'glove':
            vectors += context.weight
        if subwords is not None:
            # Ignore hash buckets absent from the training vocabulary.
            used = set(subwords.flatten().tolist())
            for word in oov:
                pieces = [piece for piece in character_ngrams(word) if piece in used]
                oov[word] = source.weight[pieces].mean(dim=0) if pieces else torch.zeros(DIMENSION)
        return F.normalize(vectors, dim=1), (source.weight.detach().clone() if subwords is not None else None), (sorted(used) if subwords is not None else [])

def encode(rows, vocabulary, unknown_id):
    tokens = torch.zeros((len(rows), MAX_LENGTH), dtype=torch.long)
    labels = torch.tensor([label for _, label in rows], dtype=torch.long)
    for index, (review, _) in enumerate(rows):
        selected = review[:MAX_LENGTH]
        tokens[index, :len(selected)] = torch.tensor([vocabulary.get(word, unknown_id) for word in selected])
    return TensorDataset(tokens, labels)

class DPCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.region = nn.Conv1d(DIMENSION,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2)
        self.blocks = nn.ModuleList(nn.Sequential(nn.ReLU(),nn.Conv1d(CHANNELS,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2),nn.ReLU(),nn.Conv1d(CHANNELS,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2)) for _ in range(BLOCKS))
        self.output = nn.Linear(CHANNELS,4)
    def forward(self,x,mask):
        x = self.region(x.transpose(1,2))
        mask = mask[:,None,:]
        x = x.masked_fill(~mask,0)
        for i,block in enumerate(self.blocks):
            if i:
                x = F.max_pool1d(x.masked_fill(~mask,-1e9),3,2,1)
                mask = F.max_pool1d(mask.float(),3,2,1).bool()
                x = x.masked_fill(~mask,0)
            x = (x + block(x)).masked_fill(~mask,0)
        return self.output(x.masked_fill(~mask,-1e9).amax(2))

def label_of(rating):
    assert 1 <= rating <= 10
    return 0 if rating <= 3 else 1 if rating <= 6 else 2 if rating <= 8 else 3

def load_inputs(directory):
    manifest=json.loads((directory/'manifest.json').read_text(encoding='utf-8'))
    if manifest['pipeline']!='latest_clean_data_64_16_20_v2':
        raise ValueError('Use the data exported by 00_prepare_data.ipynb.')
    splits, ids = {}, {}
    for split in ('train','validation'):
        info=manifest['split_info'][split]
        path=directory/info['file']
        if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Input hash mismatch: '+split)
        records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
        assert len(records)==info['n'] and records
        splits[split]=[]; ids[split]=[]
        for row in records:
            words=row['filtered_words']
            assert isinstance(words,list) and all(isinstance(w,str) and w for w in words)
            # Keep empty reviews in the same partition with an explicit model-input placeholder.
            splits[split].append((words or ['<empty>'],label_of(row['rating'])))
            ids[split].append(row['row_id'])
        assert set(y for _,y in splits[split])==set(range(4)), 'A partition is missing a class.'
    if any(manifest['normalized_text_group_overlap'].values()):
        print('NOTICE: supplied random split has shared normalized text groups:',manifest['normalized_text_group_overlap'])
    return splits,ids,manifest

def fit_vectors(data,manifest,cache_dir,epochs):
    # Cache key binds to the NEW data and every embedding-training setting.
    signature=dict(train=manifest['split_info']['train']['sha256'],validation=manifest['split_info']['validation']['sha256'],
                   method=EMBEDDING_METHOD,seed=SEED,dim=DIMENSION,window=WINDOW,buckets=BUCKETS,
                   min_count=MIN_COUNT,max_vocab=MAX_VOCAB,epochs=epochs,implementation='standalone_v2')
    digest=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    cache_dir.mkdir(parents=True,exist_ok=True)
    path=cache_dir/(EMBEDDING_METHOD+'_'+digest+'.pt')
    if path.exists():
        result=torch.load(path,weights_only=True)
        assert result['signature']==signature
        return result
    frequency=Counter(w for words,_ in data['train'] for w in words)
    words=[w for w,n in frequency.most_common() if n>=MIN_COUNT][:MAX_VOCAB]
    if not words: raise ValueError('Insufficient training vocabulary.')
    vocab={w:i for i,w in enumerate(words,1)}
    pairs,counts=make_pairs([[vocab.get(w,0) for w in row] for row,_ in data['train']])
    if pairs.numel()==0: raise ValueError('No training co-occurrence pairs.')
    oov={}
    if EMBEDDING_METHOD=='fasttext':
        # These strings are only encoded with training-learned subword weights, never fitted on validation.
        oov=dict.fromkeys(sorted({w for rows in data.values() for row,_ in rows for w in row[:MAX_LENGTH] if w not in vocab}))
    vectors,subword_source,used_subwords=train_embedding(EMBEDDING_METHOD,words,pairs,counts,
                            torch.tensor([0,*[frequency[w] for w in words]],dtype=torch.float32),oov,epochs)
    unknown=len(words)+1
    vectors=torch.cat([vectors,torch.zeros(1,DIMENSION)])
    if oov:
        vocab.update((word,unknown+1+i) for i,word in enumerate(oov))
        vectors=torch.cat([vectors,F.normalize(torch.stack(list(oov.values())),dim=1)])
    result=dict(vectors=vectors,vocab=vocab,unknown=unknown,signature=signature,subword_source=subword_source,used_subwords=used_subwords)
    temporary=path.with_suffix('.tmp')
    torch.save(result,temporary); temporary.replace(path)
    return result

def evaluate(model,embedding,loader):
    model.eval()
    probability=[]; truth=[]
    with torch.no_grad():
        for ids,y in loader:
            probability.append(model(embedding(ids),ids.ne(0)).softmax(1).numpy())
            truth.extend(y.tolist())
    target=np.asarray(truth); probability=np.concatenate(probability)
    predicted=probability.argmax(1)
    report=classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    metrics=dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                 precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                 classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist())
    return metrics,target,probability

In [ ]:
def main(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--data-dir',type=Path,default=PACKAGE_DIR/'data_64_16_20')
    parser.add_argument('--output-root',type=Path,default=PACKAGE_DIR/'results')
    parser.add_argument('--epochs',type=int,default=EPOCHS)
    parser.add_argument('--embedding-epochs',type=int,default=EMBEDDING_EPOCHS)
    parser.add_argument('--weight-alpha',type=float,choices=[0.0,0.5,1.0],default=WEIGHT_ALPHA,
                        help='0=unweighted; 0.5=square-root; 1=inverse. MUST be identical in all six combinations.')
    args=parser.parse_args(argv)
    assert args.epochs>0 and args.embedding_epochs>0
    assert KERNEL_SIZE>0 and KERNEL_SIZE%2==1 and BLOCKS>0 and RNN_HIDDEN>0
    torch.set_num_threads(4)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    data,row_ids,manifest=load_inputs(args.data_dir)
    cached=fit_vectors(data,manifest,args.data_dir/'embedding_cache',args.embedding_epochs)
    encoded={s:encode(rows,cached['vocab'],cached['unknown']) for s,rows in data.items()}
    torch.manual_seed(SEED)
    model=DPCNN()
    embedding=nn.Embedding.from_pretrained(cached['vectors'],freeze=True,padding_idx=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    generator=torch.Generator().manual_seed(SEED)
    training=DataLoader(encoded['train'],batch_size=BATCH_SIZE,shuffle=True,generator=generator)
    validation=DataLoader(encoded['validation'],batch_size=256)
    counts=torch.bincount(encoded['train'].tensors[1],minlength=4).float()
    weights=(counts.sum()/(4*counts)).pow(args.weight_alpha); weights/=weights.mean()
    run=args.output_root/(MODEL_NAME+'_'+EMBEDDING_METHOD+('_sqrt' if args.weight_alpha==0.5 else '_unweighted' if args.weight_alpha==0 else '_inverse'))/(str(time.time_ns()))
    run.mkdir(parents=True,exist_ok=False)
    config=dict(model=MODEL_NAME,embedding=EMBEDDING_METHOD,labels=dict(zip(NAMES,[[1,2,3],[4,5,6],[7,8],[9,10]])),
                input_manifest=manifest,seed=SEED,epochs=args.epochs,embedding_epochs=args.embedding_epochs,
                train_n=len(data['train']),validation_n=len(data['validation']),class_counts=counts.tolist(),
                embedding_dimension=DIMENSION,max_tokens=MAX_LENGTH,embedding_frozen=True,batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,gradient_clip=GRADIENT_CLIP,weight_alpha=args.weight_alpha,weights=weights.tolist(),
                architecture=dict(DIMENSION=DIMENSION,CHANNELS=CHANNELS,BLOCKS=BLOCKS,KERNEL_SIZE=KERNEL_SIZE,RNN_HIDDEN=RNN_HIDDEN,BUCKETS=BUCKETS),selection='best validation macro_f1',independent_test_evaluated=False,torch_version=str(torch.__version__))
    (run/'config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
    history=[]; best=-1
    for epoch in range(1,args.epochs+1):
        model.train(); numerator=denominator=0.; start=time.perf_counter()
        for ids,y in training:
            optimizer.zero_grad()
            loss=F.cross_entropy(model(embedding(ids),ids.ne(0)),y,weight=weights)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRADIENT_CLIP); optimizer.step()
            mass=weights[y].sum().item(); numerator+=loss.item()*mass; denominator+=mass
        metrics,target,probability=evaluate(model,embedding,validation)
        row=dict(epoch=epoch,train_loss=numerator/denominator,seconds=time.perf_counter()-start,**metrics)
        history.append(row)
        if row['macro_f1']>best:
            best=row['macro_f1']
            torch.save(dict(model_state=model.state_dict(),vectors=cached['vectors'],vocab=cached['vocab'],
                            unknown=cached['unknown'],config=config,subword_source=cached['subword_source'],used_subwords=cached['used_subwords']),run/'best_model.pt')
            np.savez_compressed(run/'validation_predictions.npz',row_id=np.asarray(row_ids['validation']),target=target,probability=probability)
        (run/'history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
        print(MODEL_NAME,EMBEDDING_METHOD,'epoch',epoch,{k:round(row[k],4) for k in ('accuracy','macro_f1','precision_macro','recall_macro')},flush=True)
    selected=max(history,key=lambda r:r['macro_f1'])
    checkpoint=torch.load(run/'best_model.pt',weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    verified,_,_=evaluate(model,embedding,validation)
    assert all(np.isclose(verified[k],selected[k],atol=1e-8,rtol=0) for k in ('accuracy','macro_f1','precision_macro','recall_macro'))
    (run/'best_metrics.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')
    print('Verified results saved to',run.resolve())
    return run

In [ ]:
def evaluate_final_test(run_directory, data_directory):
    # Call only after all tuning choices have been finalized.
    checkpoint=torch.load(Path(run_directory)/'best_model.pt',weights_only=True)
    cfg=checkpoint['config']
    if cfg['model']!=MODEL_NAME or cfg['embedding']!=EMBEDDING_METHOD:
        raise ValueError('Checkpoint belongs to a different combination.')
    globals().update(cfg['architecture'])
    globals()['MAX_LENGTH']=cfg['max_tokens']
    manifest=json.loads((Path(data_directory)/'manifest.json').read_text(encoding='utf-8'))
    if manifest!=cfg['input_manifest']: raise ValueError('Checkpoint and data manifest differ.')
    info=manifest['split_info']['test']; path=Path(data_directory)/info['file']
    if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']: raise ValueError('Test hash mismatch.')
    records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    assert len(records)==info['n'] and records
    rows=[(r['filtered_words'] or ['<empty>'],label_of(r['rating'])) for r in records]
    vocab=dict(checkpoint['vocab']); vectors=checkpoint['vectors']
    if EMBEDDING_METHOD=='fasttext':
        # Encode unseen TEST words using stored training-learned subword weights only.
        source=checkpoint['subword_source']; used=set(checkpoint['used_subwords'])
        unseen=sorted({w for words,_ in rows for w in words[:MAX_LENGTH] if w not in vocab})
        additions=[]
        for word in unseen:
            pieces=[p for p in character_ngrams(word) if p in used]
            value=source[pieces].mean(0) if pieces else torch.zeros(DIMENSION)
            vocab[word]=len(vectors)+len(additions); additions.append(value)
        if additions: vectors=torch.cat([vectors,F.normalize(torch.stack(additions),dim=1)])
    model=DPCNN(); model.load_state_dict(checkpoint['model_state'])
    embedding=nn.Embedding.from_pretrained(vectors,freeze=True,padding_idx=0)
    loader=DataLoader(encode(rows,vocab,checkpoint['unknown']),batch_size=256)
    metrics,target,probability=evaluate(model,embedding,loader)
    destination=Path(run_directory)/'final_test_metrics.json'
    destination.write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    np.savez_compressed(Path(run_directory)/'final_test_predictions.npz',row_id=np.asarray([r['row_id'] for r in records]),target=target,probability=probability)
    print('FINAL TEST:', {k:metrics[k] for k in ('accuracy','macro_f1','precision_macro','recall_macro')})
    return metrics

In [ ]:
assert (PACKAGE_DIR / 'data_64_16_20' / 'manifest.json').is_file(), f'請檢查資料目錄：{PACKAGE_DIR}'
run_directory = main([])

## Section 6. DPCNN with GloVe

Train the convolutional model with GloVe embeddings and square-root class weights.

Run from the repository root.

[Standalone notebook](models/DPCNN_GloVe.ipynb)

# DPCNN + GloVe — Square-Root Weighting
No terminal is required. Run 00_prepare_data.ipynb first, then run this notebook from top to bottom.
The first cell contains parameters, the second contains the training implementation and usually needs no changes, and the third starts training.
Training fits the embeddings and model, validation selects the epoch, and test evaluation is enabled only for the final independent test.
The two models with the same embedding share fixed vectors; all six combinations must use the same inputs, processing, and comparison rules.


In [ ]:
# Independent model program: no imports from the old experiment workspace.

import argparse, hashlib, json, random, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix
import seaborn  as sns
import matplotlib.pyplot as plt

MODEL_NAME = 'DPCNN'
EMBEDDING_METHOD = 'glove'
# ===== Parameter Area: Usually the Only Section to Modify =====
SEED = 42
DIMENSION = 48          # Embedding dimension: keep identical across all six combinations
MAX_LENGTH = 96         # Maximum words per text: keep identical across all six combinations
WINDOW = 4             # Embedding context window
MAX_VOCAB = 30000      # Vocabulary size

EPOCHS = 5           # Maximum training epochs; select the epoch by validation Macro-F1
EMBEDDING_EPOCHS = 5    # Number of embedding training epochs
BATCH_SIZE = 128     # Batch size
LEARNING_RATE = 0.0005     # Learning rate
WEIGHT_DECAY = 0.01       # Weight decay
GRADIENT_CLIP = 5         # Gradient norm clipping
MIN_COUNT = 2          # Minimum word frequency for the vocabulary
CHANNELS = 128          # DPCNN convolution channels / TextRCNN projection dimension
BLOCKS = 4             # DPCNN only: number of residual blocks
KERNEL_SIZE = 3        # DPCNN only: positive odd-sized convolution kernel
WEIGHT_ALPHA = 0.5  # 0=unweighted; 0.5=square-root weighting
# ===== End of Parameter Area =====
NAMES = ['low', 'moderate', 'good', 'excellent']
PACKAGE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()




Utility Functions

In [ ]:
"""Map the original 1-10 rating to a four-class label."""
def label_of(rating):
    assert 1 <= rating <= 10
    return 0 if rating <= 3 else 1 if rating <= 6 else 2 if rating <= 8 else 3

"""Convert text word lists into token-index tensors and build the dataset."""
def encode(rows, vocabulary, unknown_id):
    tokens = torch.zeros((len(rows), MAX_LENGTH), dtype=torch.long)
    labels = torch.tensor([label for _, label in rows], dtype=torch.long)
    for index, (review, _) in enumerate(rows):
        selected = review[:MAX_LENGTH]
        tokens[index, :len(selected)] = torch.tensor([vocabulary.get(word, unknown_id) for word in selected])
    return TensorDataset(tokens, labels)

"""Evaluate the model: calculate accuracy, precision, F1, recall, and other metrics."""
def evaluate(model,embedding,loader):
    model.eval()
    probability=[]; truth=[]
    with torch.no_grad():
        for ids,y in loader:
            probability.append(model(embedding(ids),ids.ne(0)).softmax(1).numpy())
            truth.extend(y.tolist())
    target=np.asarray(truth); probability=np.concatenate(probability)
    predicted=probability.argmax(1)
    report=classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    metrics=dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                 precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                 classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist())
    return metrics,target,probability

GloVe Embedding Implementation

In [ ]:
def make_pairs(corpus):
    counts = Counter()
    for sentence in corpus:
        for i, center in enumerate(sentence):
            if center == 0:
                continue
            for j in range(max(0, i - WINDOW), min(len(sentence), i + WINDOW + 1)):
                if i != j and sentence[j] != 0:
                    counts[center, sentence[j]] += 1.0 / abs(i - j)
    pairs = np.asarray(list(counts), dtype=np.int64)
    values = np.fromiter(counts.values(), dtype=np.float32, count=len(counts))
    return torch.from_numpy(pairs), torch.from_numpy(values)

def train_embedding(words, pairs, counts, epochs):
    torch.manual_seed(SEED)
    size = len(words) + 1
    # Standard GloVe two-vector representation: center word + context word
    source = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    context = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    nn.init.uniform_(source.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    nn.init.uniform_(context.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    # GloVe-specific bias terms
    source_bias = nn.Embedding(size, 1, sparse=True)
    context_bias = nn.Embedding(size, 1, sparse=True)
    nn.init.zeros_(source_bias.weight)
    nn.init.zeros_(context_bias.weight)
    parameters = [
        *source.parameters(), *context.parameters(),
        *source_bias.parameters(), *context_bias.parameters()
    ]
    optimizer = torch.optim.SparseAdam(parameters, lr=0.025)
    def centers(ids):
        return source(ids)
    for epoch in range(epochs):
        order = torch.randperm(len(pairs))  # Randomly shuffle all GloVe co-occurrence pairs
        loss_sum = 0.0
        for batch in order.split(4096):
            left, right = pairs[batch].T
            vectors = centers(left)
            positive = (vectors * context(right)).sum(dim=1)
            # GloVe weighted squared loss
            weights = counts[batch]
            residual = positive + source_bias(left).flatten() + context_bias(right).flatten() - weights.log()
            loss = (((weights / 10).clamp(max=1).pow(0.75)) * residual.square()).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f'glove embedding epoch {epoch + 1}/{epochs} loss={loss_sum / len(order):.4f}', flush=True)
    # GloVe convention: final word vector = center-word vector + context-word vector, then normalize
    with torch.no_grad():
        vectors = torch.cat([centers(ids) for ids in torch.arange(size).split(1024)])
        vectors += context.weight
        return F.normalize(vectors, dim=1)


DPCNN Classification Model

In [ ]:
class DPCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.region = nn.Conv1d(DIMENSION,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2)
        self.blocks = nn.ModuleList(nn.Sequential(nn.ReLU(),nn.Conv1d(CHANNELS,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2),nn.ReLU(),nn.Conv1d(CHANNELS,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2)) for _ in range(BLOCKS))
        self.output = nn.Linear(CHANNELS,4)
    def forward(self,x,mask):
        x = self.region(x.transpose(1,2))
        mask = mask[:,None,:]
        x = x.masked_fill(~mask,0)
        for i,block in enumerate(self.blocks):
            if i:
                x = F.max_pool1d(x.masked_fill(~mask,-1e9),3,2,1)
                mask = F.max_pool1d(mask.float(),3,2,1).bool()
                x = x.masked_fill(~mask,0)
            x = (x + block(x)).masked_fill(~mask,0)
        return self.output(x.masked_fill(~mask,-1e9).amax(2))

Main Workflow: GloVe Embedding + DPCNN Model Training

In [ ]:
def load_inputs(directory):
    manifest=json.loads((directory/'manifest.json').read_text(encoding='utf-8'))
    if manifest['pipeline']!='latest_clean_data_64_16_20_v2':
        raise ValueError('Use the data exported by 00_prepare_data.ipynb.')
    splits, ids = {}, {}
    for split in ('train','validation'):
        info=manifest['split_info'][split]
        path=directory/info['file']
        if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Input hash mismatch: '+split)
        records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
        assert len(records)==info['n'] and records
        splits[split]=[]; ids[split]=[]
        for row in records:
            words=row['filtered_words']
            assert isinstance(words,list) and all(isinstance(w,str) and w for w in words)
            # Keep empty reviews in the same partition with an explicit model-input placeholder.
            splits[split].append((words or ['<empty>'],label_of(row['rating'])))
            ids[split].append(row['row_id'])
        assert set(y for _,y in splits[split])==set(range(4)), 'A partition is missing a class.'
    if any(manifest['normalized_text_group_overlap'].values()):
        print('NOTICE: supplied random split has shared normalized text groups:',manifest['normalized_text_group_overlap'])
    return splits,ids,manifest

def fit_vectors(data, manifest, cache_dir, epochs):
    # Cache signature: bind data and training configuration to ensure reproducibility
    signature = dict(
        train=manifest['split_info']['train']['sha256'],
        validation=manifest['split_info']['validation']['sha256'],
        method=EMBEDDING_METHOD,
        seed=SEED,
        dim=DIMENSION,
        window=WINDOW,
        min_count=MIN_COUNT,
        max_vocab=MAX_VOCAB,
        epochs=epochs,
        implementation='standalone_v2'
    )
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / (EMBEDDING_METHOD + '_' + digest + '.pt')

    # Return the cached result directly when the cache is hit
    if path.exists():
        result = torch.load(path, weights_only=True)
        assert result['signature'] == signature
        return result

    # 1. Build the vocabulary from the training set
    frequency = Counter(w for words, _ in data['train'] for w in words)
    words = [w for w, n in frequency.most_common() if n >= MIN_COUNT][:MAX_VOCAB]
    if not words:
        raise ValueError('Insufficient training vocabulary.')
    vocab = {w: i for i, w in enumerate(words, 1)}

    # 2. Build GloVe co-occurrence pairs
    pairs, counts = make_pairs([[vocab.get(w, 0) for w in row] for row, _ in data['train']])
    if pairs.numel() == 0:
        raise ValueError('No training co-occurrence pairs.')

    # 3. Train GloVe word vectors (compatible with the simplified function signature)
    vectors = train_embedding(words, pairs, counts, epochs)

    # 4. Add an unknown placeholder vector (zero vector)
    unknown = len(words) + 1
    vectors = torch.cat([vectors, torch.zeros(1, DIMENSION)])

    # 5. Assemble the result and save it to the cache
    result = dict(
        vectors=vectors,
        vocab=vocab,
        unknown=unknown,
        signature=signature
    )
    temporary = path.with_suffix('.tmp')
    torch.save(result, temporary)
    temporary.replace(path)
    return result



def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-dir', type=Path, default=PACKAGE_DIR / 'data_64_16_20')
    parser.add_argument('--output-root', type=Path, default=PACKAGE_DIR / 'results')
    parser.add_argument('--epochs', type=int, default=EPOCHS)
    parser.add_argument('--embedding-epochs', type=int, default=EMBEDDING_EPOCHS)
    parser.add_argument('--weight-alpha', type=float, choices=[0.0, 0.5, 1.0], default=WEIGHT_ALPHA,
                        help='0=unweighted; 0.5=square-root; 1=inverse. MUST be identical in all six combinations.')
    args = parser.parse_args(argv)

    # Parameter validation: remove TextRCNN-specific RNN_HIDDEN validation
    assert args.epochs > 0 and args.embedding_epochs > 0
    assert KERNEL_SIZE > 0 and KERNEL_SIZE % 2 == 1 and BLOCKS > 0

    torch.set_num_threads(4)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # 1. Load the dataset
    data, row_ids, manifest = load_inputs(args.data_dir)

    # 2. Train or cache GloVe word vectors
    cached = fit_vectors(data, manifest, args.data_dir / 'embedding_cache', args.embedding_epochs)
    encoded = {s: encode(rows, cached['vocab'], cached['unknown']) for s, rows in data.items()}

    # 3. Initialize the DPCNN model and frozen embedding layer
    torch.manual_seed(SEED)
    model = DPCNN()
    embedding = nn.Embedding.from_pretrained(cached['vectors'], freeze=True, padding_idx=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # 4. Build the data loaders
    generator = torch.Generator().manual_seed(SEED)
    training = DataLoader(encoded['train'], batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    validation = DataLoader(encoded['validation'], batch_size=256)

    # 5. Calculate class-weighted loss weights
    counts = torch.bincount(encoded['train'].tensors[1], minlength=4).float()
    weights = (counts.sum() / (4 * counts)).pow(args.weight_alpha)
    weights /= weights.mean()

    # 6. Create the experiment output directory
    weight_suffix = '_sqrt' if args.weight_alpha == 0.5 else '_unweighted' if args.weight_alpha == 0 else '_inverse'
    run = args.output_root / (MODEL_NAME + '_' + EMBEDDING_METHOD + weight_suffix) / str(time.time_ns())
    run.mkdir(parents=True, exist_ok=False)

    # 7. Save the experiment configuration; remove redundant fields such as RNN_HIDDEN and BUCKETS
    config = dict(
        model=MODEL_NAME,
        embedding=EMBEDDING_METHOD,
        labels=dict(zip(NAMES, [[1, 2, 3], [4, 5, 6], [7, 8], [9, 10]])),
        input_manifest=manifest,
        seed=SEED,
        epochs=args.epochs,
        embedding_epochs=args.embedding_epochs,
        train_n=len(data['train']),
        validation_n=len(data['validation']),
        class_counts=counts.tolist(),
        embedding_dimension=DIMENSION,
        max_tokens=MAX_LENGTH,
        embedding_frozen=True,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        gradient_clip=GRADIENT_CLIP,
        weight_alpha=args.weight_alpha,
        weights=weights.tolist(),
        architecture=dict(
            DIMENSION=DIMENSION,
            CHANNELS=CHANNELS,
            BLOCKS=BLOCKS,
            KERNEL_SIZE=KERNEL_SIZE
        ),
        selection='best validation macro_f1',
        independent_test_evaluated=False,
        torch_version=str(torch.__version__)
    )
    (run / 'config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')

    # 8. Training loop
    history = []
    best = -1
    for epoch in range(1, args.epochs + 1):
        model.train()
        numerator = denominator = 0.
        start = time.perf_counter()

        for ids, y in training:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(embedding(ids), ids.ne(0)), y, weight=weights)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            mass = weights[y].sum().item()
            numerator += loss.item() * mass
            denominator += mass

        # Validation-set evaluation
        metrics, target, probability = evaluate(model, embedding, validation)
        row = dict(epoch=epoch, train_loss=numerator / denominator, seconds=time.perf_counter() - start, **metrics)
        history.append(row)

        # Save the best model; remove redundant subword_source and used_subwords fields
        if row['macro_f1'] > best:
            best = row['macro_f1']
            torch.save(dict(
                model_state=model.state_dict(),
                vectors=cached['vectors'],
                vocab=cached['vocab'],
                unknown=cached['unknown'],
                config=config
            ), run / 'best_model.pt')
            np.savez_compressed(
                run / 'validation_predictions.npz',
                row_id=np.asarray(row_ids['validation']),
                target=target,
                probability=probability
            )

        (run / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
        print(MODEL_NAME, EMBEDDING_METHOD, 'epoch', epoch,
              {k: round(row[k], 4) for k in ('accuracy', 'macro_f1', 'precision_macro', 'recall_macro')},
              flush=True)


    # 9. Verify consistency of the best-model results
    selected = max(history, key=lambda r: r['macro_f1'])
    
    checkpoint = torch.load(run / 'best_model.pt', weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    verified, _, _ = evaluate(model, embedding, validation)
    assert all(np.isclose(verified[k], selected[k], atol=1e-8, rtol=0)
               for k in ('accuracy', 'macro_f1', 'precision_macro', 'recall_macro'))

    (run / 'best_metrics.json').write_text(json.dumps(selected, indent=2), encoding='utf-8')
    print('Verified results saved to', run.resolve())
    return run



def evaluate_final_test(run_directory, data_directory):
    # Call only after all tuning is complete for final test-set evaluation
    checkpoint = torch.load(Path(run_directory) / 'best_model.pt', weights_only=True)
    cfg = checkpoint['config']

    # Verify checkpoint ownership to prevent using the wrong model
    if cfg['model'] != MODEL_NAME or cfg['embedding'] != EMBEDDING_METHOD:
        raise ValueError('Checkpoint belongs to a different combination.')

    # Restore model hyperparameters from the configuration
    globals().update(cfg['architecture'])
    globals()['MAX_LENGTH'] = cfg['max_tokens']

    # Load and verify the data manifest
    manifest = json.loads((Path(data_directory) / 'manifest.json').read_text(encoding='utf-8'))
    if manifest != cfg['input_manifest']:
        raise ValueError('Checkpoint and data manifest differ.')

    # Load the test set and verify file integrity
    info = manifest['split_info']['test']
    path = Path(data_directory) / info['file']
    if hashlib.sha256(path.read_bytes()).hexdigest() != info['sha256']:
        raise ValueError('Test hash mismatch.')

    records = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    assert len(records) == info['n'] and records

    # Text and label processing
    rows = [(r['filtered_words'] or ['<empty>'], label_of(r['rating'])) for r in records]

    # Use the trained vocabulary and vectors directly; GloVe has no subword OOV handling, so unknown words map to unknown
    vocab = dict(checkpoint['vocab'])
    vectors = checkpoint['vectors']

    # Initialize the model and embedding layer
    model = DPCNN()
    model.load_state_dict(checkpoint['model_state'])
    embedding = nn.Embedding.from_pretrained(vectors, freeze=True, padding_idx=0)

    # Build the test-set data loader and evaluate
    loader = DataLoader(encode(rows, vocab, checkpoint['unknown']), batch_size=256)
    metrics, target, probability = evaluate(model, embedding, loader)

    # Save the results
    destination = Path(run_directory) / 'final_test_metrics.json'
    destination.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    np.savez_compressed(
        Path(run_directory) / 'final_test_predictions.npz',
        row_id=np.asarray([r['row_id'] for r in records]),
        target=target,
        probability=probability
    )

    print('FINAL TEST:', {k: metrics[k] for k in ('accuracy', 'macro_f1', 'precision_macro', 'recall_macro')})
    return metrics



In [ ]:
# Change this to the folder actually exported by the data-preparation notebook (if it is not the current directory)
DATA_DIR = PACKAGE_DIR / 'data_64_16_20'
# Pass arguments directly to main and start training
run_directory = main(['--data-dir', str(DATA_DIR)])


In [ ]:
# Keep this False during tuning. Set it to True only after all settings are finalized to evaluate the saved best model.
RUN_FINAL_TEST = True
if RUN_FINAL_TEST:
    final_test_results = evaluate_final_test(run_directory, DATA_DIR)


## Section 7. DPCNN with SGNS

Use the selected version 3 configuration and square-root class weights.

Run from the repository root.

[Standalone notebook](models/DPCNN_SGNS.ipynb)

# DPCNN + sgns — 平方根加权
无需终端。先运行 00_prepare_data.ipynb，再从上到下运行本 notebook。
第一格为参数区；第二格为训练实现，通常不需要修改；第三格开始训练。
train 拟合 embedding 和模型，validation 选轮次，test 仅在最后主动开启评估。
相同 embedding 的两个模型共享固定向量；六组的输入、处理和比较规则必须统一。


In [ ]:
MODEL_NAME = 'DPCNN'
EMBEDDING_METHOD = 'sgns'
# ===== 参数区：通常只需修改这里 =====
#相对ver1修改以下参数：
#Epochs 5 -> 6
#Weight_decay 0.01 -> 0.02
SEED = 42
DIMENSION = 48          # embedding 维数：六组保持一致
MAX_LENGTH = 96         # 文本最大词数：六组保持一致
WINDOW = 4              # embedding 上下文窗口
BUCKETS = 50000         # FastText 子词哈希桶数
EPOCHS = 6              # 最大训练轮数，按 validation Macro-F1 选最好一轮
EMBEDDING_EPOCHS = 2    # embedding 训练轮数
BATCH_SIZE = 128
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.02
GRADIENT_CLIP = 5
MIN_COUNT = 2
MAX_VOCAB = 30000
CHANNELS = 64           # DPCNN 卷积通道 / TextRCNN 投影维度
BLOCKS = 6              # 仅 DPCNN：残差块数
KERNEL_SIZE = 3         # 仅 DPCNN：正奇数卷积核
RNN_HIDDEN = 32         # 仅 TextRCNN：每个方向的 LSTM 隐层维数
WEIGHT_ALPHA = 0.5      # 0=普通版；0.5=平方根加权版
# ===== 参数区结束 =====

In [ ]:
import argparse
import hashlib
import json
import random
import time
import zlib
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, TensorDataset

PACKAGE_DIR = Path.cwd()
NAMES = ['low_1_3', 'moderate_4_6', 'good_7_8', 'excellent_9_10']

def character_ngrams(word):
    bounded = f'<{word}>'
    return [1 + zlib.crc32(bounded[i:i+n].encode()) % BUCKETS
            for n in range(3, 7) for i in range(len(bounded) - n + 1)
            if n != len(bounded)]

def component_table(words):
    rows = [[]] + [[BUCKETS + index, *character_ngrams(word)]
                   for index, word in enumerate(words, start=1)]
    table = torch.zeros((len(rows), max(map(len, rows))), dtype=torch.long)
    for index, row in enumerate(rows):
        table[index, :len(row)] = torch.tensor(row, dtype=torch.long)
    return table

def make_pairs(corpus):
    counts = Counter()
    for sentence in corpus:
        for i, center in enumerate(sentence):
            if center == 0:
                continue
            for j in range(max(0, i - WINDOW), min(len(sentence), i + WINDOW + 1)):
                if i != j and sentence[j] != 0:
                    counts[center, sentence[j]] += 1.0 / abs(i - j)
    pairs = np.asarray(list(counts), dtype=np.int64)
    values = np.fromiter(counts.values(), dtype=np.float32, count=len(counts))
    return torch.from_numpy(pairs), torch.from_numpy(values)

def train_embedding(method, words, pairs, counts, frequency, oov, epochs):
    torch.manual_seed(SEED)
    size = len(words) + 1
    subwords = component_table(words) if method == 'fasttext' else None
    input_size = size + BUCKETS if subwords is not None else size
    source = nn.Embedding(input_size, DIMENSION, padding_idx=0, sparse=True)
    context = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    nn.init.uniform_(source.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    if method == 'glove':
        nn.init.uniform_(context.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
        source_bias, context_bias = nn.Embedding(size, 1, sparse=True), nn.Embedding(size, 1, sparse=True)
        nn.init.zeros_(source_bias.weight)
        nn.init.zeros_(context_bias.weight)
        parameters = [*source.parameters(), *context.parameters(), *source_bias.parameters(), *context_bias.parameters()]
    else:
        nn.init.zeros_(context.weight)
        parameters = [*source.parameters(), *context.parameters()]
    optimizer = torch.optim.SparseAdam(parameters, lr=0.025 if method == 'glove' else 0.01)
    noise = frequency.pow(0.75)

    def centers(ids):
        if subwords is None:
            return source(ids)
        components = subwords[ids]
        return source(components).sum(dim=1) / components.ne(0).sum(dim=1, keepdim=True).clamp_min(1)

    for epoch in range(epochs):
        if method == 'glove':
            order = torch.randperm(len(pairs))
        else:
            # Sample observed pairs by distance-weighted frequency for SGNS.
            order = torch.multinomial(counts, len(pairs), replacement=True)
        loss_sum = 0.0
        for batch in order.split(4096):
            left, right = pairs[batch].T
            vectors = centers(left)
            positive = (vectors * context(right)).sum(dim=1)
            if method == 'glove':
                weights = counts[batch]
                residual = positive + source_bias(left).flatten() + context_bias(right).flatten() - weights.log()
                loss = (((weights / 10).clamp(max=1).pow(0.75)) * residual.square()).mean()
            else:
                negatives = torch.multinomial(noise, len(batch) * 5, replacement=True).reshape(-1, 5)
                negative = torch.einsum('bd,bkd->bk', vectors, context(negatives))
                loss = -(F.logsigmoid(positive) + F.logsigmoid(-negative).sum(dim=1)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f'{method} embedding epoch {epoch + 1}/{epochs} loss={loss_sum / len(order):.4f}', flush=True)

    with torch.no_grad():
        vectors = torch.cat([centers(ids) for ids in torch.arange(size).split(1024)])
        if method == 'glove':
            vectors += context.weight
        if subwords is not None:
            # Ignore hash buckets absent from the training vocabulary.
            used = set(subwords.flatten().tolist())
            for word in oov:
                pieces = [piece for piece in character_ngrams(word) if piece in used]
                oov[word] = source.weight[pieces].mean(dim=0) if pieces else torch.zeros(DIMENSION)
        return F.normalize(vectors, dim=1), (source.weight.detach().clone() if subwords is not None else None), (sorted(used) if subwords is not None else [])

def encode(rows, vocabulary, unknown_id):
    tokens = torch.zeros((len(rows), MAX_LENGTH), dtype=torch.long)
    labels = torch.tensor([label for _, label in rows], dtype=torch.long)
    for index, (review, _) in enumerate(rows):
        selected = review[:MAX_LENGTH]
        tokens[index, :len(selected)] = torch.tensor([vocabulary.get(word, unknown_id) for word in selected])
    return TensorDataset(tokens, labels)

class DPCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.region = nn.Conv1d(DIMENSION,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2)
        self.blocks = nn.ModuleList(nn.Sequential(nn.ReLU(),nn.Conv1d(CHANNELS,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2),nn.ReLU(),nn.Conv1d(CHANNELS,CHANNELS,KERNEL_SIZE,padding=KERNEL_SIZE//2)) for _ in range(BLOCKS))
        self.output = nn.Linear(CHANNELS,4)
    def forward(self,x,mask):
        x = self.region(x.transpose(1,2))
        mask = mask[:,None,:]
        x = x.masked_fill(~mask,0)
        for i,block in enumerate(self.blocks):
            if i:
                x = F.max_pool1d(x.masked_fill(~mask,-1e9),3,2,1)
                mask = F.max_pool1d(mask.float(),3,2,1).bool()
                x = x.masked_fill(~mask,0)
            x = (x + block(x)).masked_fill(~mask,0)
        return self.output(x.masked_fill(~mask,-1e9).amax(2))

def label_of(rating):
    assert 1 <= rating <= 10
    return 0 if rating <= 3 else 1 if rating <= 6 else 2 if rating <= 8 else 3

def load_inputs(directory):
    manifest=json.loads((directory/'manifest.json').read_text(encoding='utf-8'))
    if manifest['pipeline']!='latest_clean_data_64_16_20_v2':
        raise ValueError('Use the data exported by 00_prepare_data.ipynb.')
    splits, ids = {}, {}
    for split in ('train','validation'):
        info=manifest['split_info'][split]
        path=directory/info['file']
        if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Input hash mismatch: '+split)
        records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
        assert len(records)==info['n'] and records
        splits[split]=[]; ids[split]=[]
        for row in records:
            words=row['filtered_words']
            assert isinstance(words,list) and all(isinstance(w,str) and w for w in words)
            # Keep empty reviews in the same partition with an explicit model-input placeholder.
            splits[split].append((words or ['<empty>'],label_of(row['rating'])))
            ids[split].append(row['row_id'])
        assert set(y for _,y in splits[split])==set(range(4)), 'A partition is missing a class.'
    if any(manifest['normalized_text_group_overlap'].values()):
        print('NOTICE: supplied random split has shared normalized text groups:',manifest['normalized_text_group_overlap'])
    return splits,ids,manifest

def fit_vectors(data,manifest,cache_dir,epochs):
    # Cache key binds to the NEW data and every embedding-training setting.
    signature=dict(train=manifest['split_info']['train']['sha256'],validation=manifest['split_info']['validation']['sha256'],
                   method=EMBEDDING_METHOD,seed=SEED,dim=DIMENSION,window=WINDOW,buckets=BUCKETS,
                   min_count=MIN_COUNT,max_vocab=MAX_VOCAB,epochs=epochs,implementation='standalone_v2')
    digest=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    cache_dir.mkdir(parents=True,exist_ok=True)
    path=cache_dir/(EMBEDDING_METHOD+'_'+digest+'.pt')
    if path.exists():
        result=torch.load(path,weights_only=True)
        assert result['signature']==signature
        return result
    frequency=Counter(w for words,_ in data['train'] for w in words)
    words=[w for w,n in frequency.most_common() if n>=MIN_COUNT][:MAX_VOCAB]
    if not words: raise ValueError('Insufficient training vocabulary.')
    vocab={w:i for i,w in enumerate(words,1)}
    pairs,counts=make_pairs([[vocab.get(w,0) for w in row] for row,_ in data['train']])
    if pairs.numel()==0: raise ValueError('No training co-occurrence pairs.')
    oov={}
    if EMBEDDING_METHOD=='fasttext':
        # These strings are only encoded with training-learned subword weights, never fitted on validation.
        oov=dict.fromkeys(sorted({w for rows in data.values() for row,_ in rows for w in row[:MAX_LENGTH] if w not in vocab}))
    vectors,subword_source,used_subwords=train_embedding(EMBEDDING_METHOD,words,pairs,counts,
                            torch.tensor([0,*[frequency[w] for w in words]],dtype=torch.float32),oov,epochs)
    unknown=len(words)+1
    vectors=torch.cat([vectors,torch.zeros(1,DIMENSION)])
    if oov:
        vocab.update((word,unknown+1+i) for i,word in enumerate(oov))
        vectors=torch.cat([vectors,F.normalize(torch.stack(list(oov.values())),dim=1)])
    result=dict(vectors=vectors,vocab=vocab,unknown=unknown,signature=signature,subword_source=subword_source,used_subwords=used_subwords)
    temporary=path.with_suffix('.tmp')
    torch.save(result,temporary); temporary.replace(path)
    return result

def evaluate(model,embedding,loader):
    model.eval()
    probability=[]; truth=[]
    with torch.no_grad():
        for ids,y in loader:
            probability.append(model(embedding(ids),ids.ne(0)).softmax(1).numpy())
            truth.extend(y.tolist())
    target=np.asarray(truth); probability=np.concatenate(probability)
    predicted=probability.argmax(1)
    report=classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    metrics=dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                 precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                 classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist())
    return metrics,target,probability

def main(argv=None):
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    parser=argparse.ArgumentParser()
    parser.add_argument('--data-dir',type=Path,default=PACKAGE_DIR/'data_64_16_20')
    parser.add_argument('--output-root',type=Path,default=PACKAGE_DIR/'results')
    parser.add_argument('--epochs',type=int,default=EPOCHS)
    parser.add_argument('--embedding-epochs',type=int,default=EMBEDDING_EPOCHS)
    parser.add_argument('--weight-alpha',type=float,choices=[0.0,0.5,1.0],default=WEIGHT_ALPHA,
                        help='0=unweighted; 0.5=square-root; 1=inverse. MUST be identical in all six combinations.')
    args=parser.parse_args(argv)
    assert args.epochs>0 and args.embedding_epochs>0
    assert KERNEL_SIZE>0 and KERNEL_SIZE%2==1 and BLOCKS>0 and RNN_HIDDEN>0
    torch.set_num_threads(8)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    data,row_ids,manifest=load_inputs(args.data_dir)
    cached=fit_vectors(data,manifest,args.data_dir/'embedding_cache',args.embedding_epochs)
    encoded={s:encode(rows,cached['vocab'],cached['unknown']) for s,rows in data.items()}
    torch.manual_seed(SEED)
    model=DPCNN()
    embedding=nn.Embedding.from_pretrained(cached['vectors'],freeze=True,padding_idx=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    generator=torch.Generator().manual_seed(SEED)
    training=DataLoader(encoded['train'],batch_size=BATCH_SIZE,shuffle=True,generator=generator)
    validation=DataLoader(encoded['validation'],batch_size=256)
    counts=torch.bincount(encoded['train'].tensors[1],minlength=4).float()
    weights=(counts.sum()/(4*counts)).pow(args.weight_alpha); weights/=weights.mean()
    run=args.output_root/(MODEL_NAME+'_'+EMBEDDING_METHOD+('_sqrt' if args.weight_alpha==0.5 else '_unweighted' if args.weight_alpha==0 else '_inverse'))/(str(time.time_ns()))
    run.mkdir(parents=True,exist_ok=False)
    config=dict(model=MODEL_NAME,embedding=EMBEDDING_METHOD,labels=dict(zip(NAMES,[[1,2,3],[4,5,6],[7,8],[9,10]])),
                input_manifest=manifest,seed=SEED,epochs=args.epochs,embedding_epochs=args.embedding_epochs,
                train_n=len(data['train']),validation_n=len(data['validation']),class_counts=counts.tolist(),
                embedding_dimension=DIMENSION,max_tokens=MAX_LENGTH,embedding_frozen=True,batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,gradient_clip=GRADIENT_CLIP,weight_alpha=args.weight_alpha,weights=weights.tolist(),
                architecture=dict(DIMENSION=DIMENSION,CHANNELS=CHANNELS,BLOCKS=BLOCKS,KERNEL_SIZE=KERNEL_SIZE,RNN_HIDDEN=RNN_HIDDEN,BUCKETS=BUCKETS),selection='best validation macro_f1',independent_test_evaluated=False,torch_version=str(torch.__version__))
    (run/'config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
    history=[]; best=-1
    for epoch in range(1,args.epochs+1):
        model.train(); numerator=denominator=0.; start=time.perf_counter()
        for ids,y in training:
            optimizer.zero_grad()
            loss=F.cross_entropy(model(embedding(ids),ids.ne(0)),y,weight=weights)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRADIENT_CLIP); optimizer.step()
            mass=weights[y].sum().item(); numerator+=loss.item()*mass; denominator+=mass
        metrics,target,probability=evaluate(model,embedding,validation)
        row=dict(epoch=epoch,train_loss=numerator/denominator,seconds=time.perf_counter()-start,**metrics)
        history.append(row)
        if row['macro_f1']>best:
            best=row['macro_f1']
            torch.save(dict(model_state=model.state_dict(),vectors=cached['vectors'],vocab=cached['vocab'],
                            unknown=cached['unknown'],config=config,subword_source=cached['subword_source'],used_subwords=cached['used_subwords']),run/'best_model.pt')
            np.savez_compressed(run/'validation_predictions.npz',row_id=np.asarray(row_ids['validation']),target=target,probability=probability)
        (run/'history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
        print(MODEL_NAME,EMBEDDING_METHOD,'epoch',epoch,{k:round(row[k],4) for k in ('accuracy','macro_f1','precision_macro','recall_macro')},flush=True)
    selected=max(history,key=lambda r:r['macro_f1'])
    checkpoint=torch.load(run/'best_model.pt',weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    verified,_,_=evaluate(model,embedding,validation)
    assert all(np.isclose(verified[k],selected[k],atol=1e-8,rtol=0) for k in ('accuracy','macro_f1','precision_macro','recall_macro'))
    (run/'best_metrics.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')
    print('Verified results saved to',run.resolve())
    return run


def evaluate_final_test(run_directory, data_directory):
    # Call only after all tuning choices have been finalized.
    checkpoint=torch.load(Path(run_directory)/'best_model.pt',weights_only=True)
    cfg=checkpoint['config']
    if cfg['model']!=MODEL_NAME or cfg['embedding']!=EMBEDDING_METHOD:
        raise ValueError('Checkpoint belongs to a different combination.')
    globals().update(cfg['architecture'])
    globals()['MAX_LENGTH']=cfg['max_tokens']
    manifest=json.loads((Path(data_directory)/'manifest.json').read_text(encoding='utf-8'))
    if manifest!=cfg['input_manifest']: raise ValueError('Checkpoint and data manifest differ.')
    info=manifest['split_info']['test']; path=Path(data_directory)/info['file']
    if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']: raise ValueError('Test hash mismatch.')
    records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    assert len(records)==info['n'] and records
    rows=[(r['filtered_words'] or ['<empty>'],label_of(r['rating'])) for r in records]
    vocab=dict(checkpoint['vocab']); vectors=checkpoint['vectors']
    if EMBEDDING_METHOD=='fasttext':
        # Encode unseen TEST words using stored training-learned subword weights only.
        source=checkpoint['subword_source']; used=set(checkpoint['used_subwords'])
        unseen=sorted({w for words,_ in rows for w in words[:MAX_LENGTH] if w not in vocab})
        additions=[]
        for word in unseen:
            pieces=[p for p in character_ngrams(word) if p in used]
            value=source[pieces].mean(0) if pieces else torch.zeros(DIMENSION)
            vocab[word]=len(vectors)+len(additions); additions.append(value)
        if additions: vectors=torch.cat([vectors,F.normalize(torch.stack(additions),dim=1)])
    model=DPCNN(); model.load_state_dict(checkpoint['model_state'])
    embedding=nn.Embedding.from_pretrained(vectors,freeze=True,padding_idx=0)
    loader=DataLoader(encode(rows,vocab,checkpoint['unknown']),batch_size=256)
    metrics,target,probability=evaluate(model,embedding,loader)
    destination=Path(run_directory)/'final_test_metrics.json'
    destination.write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    np.savez_compressed(Path(run_directory)/'final_test_predictions.npz',row_id=np.asarray([r['row_id'] for r in records]),target=target,probability=probability)
    print('FINAL TEST:', {k:metrics[k] for k in ('accuracy','macro_f1','precision_macro','recall_macro')})
    return metrics


In [ ]:
# 修改为数据准备 notebook 实际导出的文件夹（如果不是当前目录）
DATA_DIR = PACKAGE_DIR / 'data_64_16_20'
run_directory = main(['--data-dir', str(DATA_DIR)])


In [ ]:
# 调参期间保持 False。所有设置确定后再改为 True，评估保存的最佳模型。
RUN_FINAL_TEST = True
if RUN_FINAL_TEST:
    final_test_results = evaluate_final_test(run_directory, DATA_DIR)


## Section 8. TextRCNN with FastText

Train the recurrent convolutional classifier with FastText embeddings and square-root class weights.

Run from the repository root.

[Standalone notebook](models/TextRCNN_FastText.ipynb)

# TextRCNN＋FastText（平方根加權）
將 notebook 放在含有 data_64_16_20 資料夾的目錄，以該目錄為工作目錄，點選 Run All。預設訓練 10 輪，按驗證 Macro-F1 保存最佳權重。

In [ ]:
# Independent model program: no imports from the old experiment workspace.
import argparse, hashlib, json, random, time, zlib
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix

MODEL_NAME = 'TextRCNN'
EMBEDDING_METHOD = 'fasttext'
# ===== 参数区：通常只需修改这里 =====
SEED = 42
DIMENSION = 48          # embedding 维数：六组保持一致
MAX_LENGTH = 96         # 文本最大词数：六组保持一致
WINDOW = 4             # embedding 上下文窗口
BUCKETS = 50000         # FastText 子词哈希桶数
EPOCHS = 10             # 最大训练轮数，按 validation Macro-F1 选最好一轮
EMBEDDING_EPOCHS = 2    # embedding 训练轮数
BATCH_SIZE = 128
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01
GRADIENT_CLIP = 5
MIN_COUNT = 2
MAX_VOCAB = 30000
CHANNELS = 128          # DPCNN 卷积通道 / TextRCNN 投影维度
BLOCKS = 6             # 仅 DPCNN：残差块数
KERNEL_SIZE = 3        # 仅 DPCNN：正奇数卷积核
RNN_HIDDEN = 128        # 仅 TextRCNN：每个方向的 LSTM 隐层维数
WEIGHT_ALPHA = 0.5  # 0=普通版；0.5=平方根加权版
# ===== 参数区结束 =====
NAMES = ['low', 'moderate', 'good', 'excellent']
PACKAGE_DIR = Path.cwd()

In [ ]:
def character_ngrams(word):
    bounded = f'<{word}>'
    return [1 + zlib.crc32(bounded[i:i+n].encode()) % BUCKETS
            for n in range(3, 7) for i in range(len(bounded) - n + 1)
            if n != len(bounded)]

def component_table(words):
    rows = [[]] + [[BUCKETS + index, *character_ngrams(word)]
                   for index, word in enumerate(words, start=1)]
    table = torch.zeros((len(rows), max(map(len, rows))), dtype=torch.long)
    for index, row in enumerate(rows):
        table[index, :len(row)] = torch.tensor(row, dtype=torch.long)
    return table

def make_pairs(corpus):
    counts = Counter()
    for sentence in corpus:
        for i, center in enumerate(sentence):
            if center == 0:
                continue
            for j in range(max(0, i - WINDOW), min(len(sentence), i + WINDOW + 1)):
                if i != j and sentence[j] != 0:
                    counts[center, sentence[j]] += 1.0 / abs(i - j)
    pairs = np.asarray(list(counts), dtype=np.int64)
    values = np.fromiter(counts.values(), dtype=np.float32, count=len(counts))
    return torch.from_numpy(pairs), torch.from_numpy(values)

def train_embedding(method, words, pairs, counts, frequency, oov, epochs):
    torch.manual_seed(SEED)
    size = len(words) + 1
    subwords = component_table(words) if method == 'fasttext' else None
    input_size = size + BUCKETS if subwords is not None else size
    source = nn.Embedding(input_size, DIMENSION, padding_idx=0, sparse=True)
    context = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    nn.init.uniform_(source.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    if method == 'glove':
        nn.init.uniform_(context.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
        source_bias, context_bias = nn.Embedding(size, 1, sparse=True), nn.Embedding(size, 1, sparse=True)
        nn.init.zeros_(source_bias.weight)
        nn.init.zeros_(context_bias.weight)
        parameters = [*source.parameters(), *context.parameters(), *source_bias.parameters(), *context_bias.parameters()]
    else:
        nn.init.zeros_(context.weight)
        parameters = [*source.parameters(), *context.parameters()]
    optimizer = torch.optim.SparseAdam(parameters, lr=0.025 if method == 'glove' else 0.01)
    noise = frequency.pow(0.75)

    def centers(ids):
        if subwords is None:
            return source(ids)
        components = subwords[ids]
        return source(components).sum(dim=1) / components.ne(0).sum(dim=1, keepdim=True).clamp_min(1)

    for epoch in range(epochs):
        if method == 'glove':
            order = torch.randperm(len(pairs))
        else:
            # Sample observed pairs by distance-weighted frequency for SGNS.
            order = torch.multinomial(counts, len(pairs), replacement=True)
        loss_sum = 0.0
        for batch in order.split(4096):
            left, right = pairs[batch].T
            vectors = centers(left)
            positive = (vectors * context(right)).sum(dim=1)
            if method == 'glove':
                weights = counts[batch]
                residual = positive + source_bias(left).flatten() + context_bias(right).flatten() - weights.log()
                loss = (((weights / 10).clamp(max=1).pow(0.75)) * residual.square()).mean()
            else:
                negatives = torch.multinomial(noise, len(batch) * 5, replacement=True).reshape(-1, 5)
                negative = torch.einsum('bd,bkd->bk', vectors, context(negatives))
                loss = -(F.logsigmoid(positive) + F.logsigmoid(-negative).sum(dim=1)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f'{method} embedding epoch {epoch + 1}/{epochs} loss={loss_sum / len(order):.4f}', flush=True)

    with torch.no_grad():
        vectors = torch.cat([centers(ids) for ids in torch.arange(size).split(1024)])
        if method == 'glove':
            vectors += context.weight
        if subwords is not None:
            # Ignore hash buckets absent from the training vocabulary.
            used = set(subwords.flatten().tolist())
            for word in oov:
                pieces = [piece for piece in character_ngrams(word) if piece in used]
                oov[word] = source.weight[pieces].mean(dim=0) if pieces else torch.zeros(DIMENSION)
        return F.normalize(vectors, dim=1), (source.weight.detach().clone() if subwords is not None else None), (sorted(used) if subwords is not None else [])

def encode(rows, vocabulary, unknown_id):
    tokens = torch.zeros((len(rows), MAX_LENGTH), dtype=torch.long)
    labels = torch.tensor([label for _, label in rows], dtype=torch.long)
    for index, (review, _) in enumerate(rows):
        selected = review[:MAX_LENGTH]
        tokens[index, :len(selected)] = torch.tensor([vocabulary.get(word, unknown_id) for word in selected])
    return TensorDataset(tokens, labels)

class TextRCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(DIMENSION,RNN_HIDDEN,batch_first=True,bidirectional=True)
        self.project = nn.Linear(DIMENSION+2*RNN_HIDDEN,CHANNELS)
        self.output = nn.Linear(CHANNELS,4)
    def forward(self,x,mask):
        packed = nn.utils.rnn.pack_padded_sequence(x,mask.sum(1).cpu(),batch_first=True,enforce_sorted=False)
        hidden,_ = self.rnn(packed)
        hidden,_ = nn.utils.rnn.pad_packed_sequence(hidden,batch_first=True,total_length=x.shape[1])
        left = F.pad(hidden[:,:-1,:RNN_HIDDEN],(0,0,1,0))
        right = F.pad(hidden[:,1:,RNN_HIDDEN:],(0,0,0,1))
        features = torch.tanh(self.project(torch.cat([left,x,right],-1)))
        return self.output(features.masked_fill(~mask[:,:,None],-1e9).amax(1))

def label_of(rating):
    assert 1 <= rating <= 10
    return 0 if rating <= 3 else 1 if rating <= 6 else 2 if rating <= 8 else 3

def load_inputs(directory):
    manifest=json.loads((directory/'manifest.json').read_text(encoding='utf-8'))
    if manifest['pipeline']!='latest_clean_data_64_16_20_v2':
        raise ValueError('Use the data exported by 00_prepare_data.ipynb.')
    splits, ids = {}, {}
    for split in ('train','validation'):
        info=manifest['split_info'][split]
        path=directory/info['file']
        if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Input hash mismatch: '+split)
        records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
        assert len(records)==info['n'] and records
        splits[split]=[]; ids[split]=[]
        for row in records:
            words=row['filtered_words']
            assert isinstance(words,list) and all(isinstance(w,str) and w for w in words)
            # Keep empty reviews in the same partition with an explicit model-input placeholder.
            splits[split].append((words or ['<empty>'],label_of(row['rating'])))
            ids[split].append(row['row_id'])
        assert set(y for _,y in splits[split])==set(range(4)), 'A partition is missing a class.'
    if any(manifest['normalized_text_group_overlap'].values()):
        print('NOTICE: supplied random split has shared normalized text groups:',manifest['normalized_text_group_overlap'])
    return splits,ids,manifest

def fit_vectors(data,manifest,cache_dir,epochs):
    # Cache key binds to the NEW data and every embedding-training setting.
    signature=dict(train=manifest['split_info']['train']['sha256'],validation=manifest['split_info']['validation']['sha256'],
                   method=EMBEDDING_METHOD,seed=SEED,dim=DIMENSION,window=WINDOW,buckets=BUCKETS,
                   min_count=MIN_COUNT,max_vocab=MAX_VOCAB,epochs=epochs,implementation='standalone_v2')
    digest=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    cache_dir.mkdir(parents=True,exist_ok=True)
    path=cache_dir/(EMBEDDING_METHOD+'_'+digest+'.pt')
    if path.exists():
        result=torch.load(path,weights_only=True)
        assert result['signature']==signature
        return result
    frequency=Counter(w for words,_ in data['train'] for w in words)
    words=[w for w,n in frequency.most_common() if n>=MIN_COUNT][:MAX_VOCAB]
    if not words: raise ValueError('Insufficient training vocabulary.')
    vocab={w:i for i,w in enumerate(words,1)}
    pairs,counts=make_pairs([[vocab.get(w,0) for w in row] for row,_ in data['train']])
    if pairs.numel()==0: raise ValueError('No training co-occurrence pairs.')
    oov={}
    if EMBEDDING_METHOD=='fasttext':
        # These strings are only encoded with training-learned subword weights, never fitted on validation.
        oov=dict.fromkeys(sorted({w for rows in data.values() for row,_ in rows for w in row[:MAX_LENGTH] if w not in vocab}))
    vectors,subword_source,used_subwords=train_embedding(EMBEDDING_METHOD,words,pairs,counts,
                            torch.tensor([0,*[frequency[w] for w in words]],dtype=torch.float32),oov,epochs)
    unknown=len(words)+1
    vectors=torch.cat([vectors,torch.zeros(1,DIMENSION)])
    if oov:
        vocab.update((word,unknown+1+i) for i,word in enumerate(oov))
        vectors=torch.cat([vectors,F.normalize(torch.stack(list(oov.values())),dim=1)])
    result=dict(vectors=vectors,vocab=vocab,unknown=unknown,signature=signature,subword_source=subword_source,used_subwords=used_subwords)
    temporary=path.with_suffix('.tmp')
    torch.save(result,temporary); temporary.replace(path)
    return result

def evaluate(model,embedding,loader):
    model.eval()
    probability=[]; truth=[]
    with torch.no_grad():
        for ids,y in loader:
            probability.append(model(embedding(ids),ids.ne(0)).softmax(1).numpy())
            truth.extend(y.tolist())
    target=np.asarray(truth); probability=np.concatenate(probability)
    predicted=probability.argmax(1)
    report=classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    metrics=dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                 precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                 classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist())
    return metrics,target,probability

In [ ]:
def main(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--data-dir',type=Path,default=PACKAGE_DIR/'data_64_16_20')
    parser.add_argument('--output-root',type=Path,default=PACKAGE_DIR/'results')
    parser.add_argument('--epochs',type=int,default=EPOCHS)
    parser.add_argument('--embedding-epochs',type=int,default=EMBEDDING_EPOCHS)
    parser.add_argument('--weight-alpha',type=float,choices=[0.0,0.5,1.0],default=WEIGHT_ALPHA,
                        help='0=unweighted; 0.5=square-root; 1=inverse. MUST be identical in all six combinations.')
    args=parser.parse_args(argv)
    assert args.epochs>0 and args.embedding_epochs>0
    assert KERNEL_SIZE>0 and KERNEL_SIZE%2==1 and BLOCKS>0 and RNN_HIDDEN>0
    torch.set_num_threads(4)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    data,row_ids,manifest=load_inputs(args.data_dir)
    cached=fit_vectors(data,manifest,args.data_dir/'embedding_cache',args.embedding_epochs)
    encoded={s:encode(rows,cached['vocab'],cached['unknown']) for s,rows in data.items()}
    torch.manual_seed(SEED)
    model=TextRCNN()
    embedding=nn.Embedding.from_pretrained(cached['vectors'],freeze=True,padding_idx=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    generator=torch.Generator().manual_seed(SEED)
    training=DataLoader(encoded['train'],batch_size=BATCH_SIZE,shuffle=True,generator=generator)
    validation=DataLoader(encoded['validation'],batch_size=256)
    counts=torch.bincount(encoded['train'].tensors[1],minlength=4).float()
    weights=(counts.sum()/(4*counts)).pow(args.weight_alpha); weights/=weights.mean()
    run=args.output_root/(MODEL_NAME+'_'+EMBEDDING_METHOD+('_sqrt' if args.weight_alpha==0.5 else '_unweighted' if args.weight_alpha==0 else '_inverse'))/(str(time.time_ns()))
    run.mkdir(parents=True,exist_ok=False)
    config=dict(model=MODEL_NAME,embedding=EMBEDDING_METHOD,labels=dict(zip(NAMES,[[1,2,3],[4,5,6],[7,8],[9,10]])),
                input_manifest=manifest,seed=SEED,epochs=args.epochs,embedding_epochs=args.embedding_epochs,
                train_n=len(data['train']),validation_n=len(data['validation']),class_counts=counts.tolist(),
                embedding_dimension=DIMENSION,max_tokens=MAX_LENGTH,embedding_frozen=True,batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,gradient_clip=GRADIENT_CLIP,weight_alpha=args.weight_alpha,weights=weights.tolist(),
                architecture=dict(DIMENSION=DIMENSION,CHANNELS=CHANNELS,BLOCKS=BLOCKS,KERNEL_SIZE=KERNEL_SIZE,RNN_HIDDEN=RNN_HIDDEN,BUCKETS=BUCKETS),selection='best validation macro_f1',independent_test_evaluated=False,torch_version=str(torch.__version__))
    (run/'config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
    history=[]; best=-1
    for epoch in range(1,args.epochs+1):
        model.train(); numerator=denominator=0.; start=time.perf_counter()
        for ids,y in training:
            optimizer.zero_grad()
            loss=F.cross_entropy(model(embedding(ids),ids.ne(0)),y,weight=weights)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRADIENT_CLIP); optimizer.step()
            mass=weights[y].sum().item(); numerator+=loss.item()*mass; denominator+=mass
        metrics,target,probability=evaluate(model,embedding,validation)
        row=dict(epoch=epoch,train_loss=numerator/denominator,seconds=time.perf_counter()-start,**metrics)
        history.append(row)
        if row['macro_f1']>best:
            best=row['macro_f1']
            torch.save(dict(model_state=model.state_dict(),vectors=cached['vectors'],vocab=cached['vocab'],
                            unknown=cached['unknown'],config=config,subword_source=cached['subword_source'],used_subwords=cached['used_subwords']),run/'best_model.pt')
            np.savez_compressed(run/'validation_predictions.npz',row_id=np.asarray(row_ids['validation']),target=target,probability=probability)
        (run/'history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
        print(MODEL_NAME,EMBEDDING_METHOD,'epoch',epoch,{k:round(row[k],4) for k in ('accuracy','macro_f1','precision_macro','recall_macro')},flush=True)
    selected=max(history,key=lambda r:r['macro_f1'])
    checkpoint=torch.load(run/'best_model.pt',weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    verified,_,_=evaluate(model,embedding,validation)
    assert all(np.isclose(verified[k],selected[k],atol=1e-8,rtol=0) for k in ('accuracy','macro_f1','precision_macro','recall_macro'))
    (run/'best_metrics.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')
    print('Verified results saved to',run.resolve())
    return run

In [ ]:
def evaluate_final_test(run_directory, data_directory):
    # Call only after all tuning choices have been finalized.
    checkpoint=torch.load(Path(run_directory)/'best_model.pt',weights_only=True)
    cfg=checkpoint['config']
    if cfg['model']!=MODEL_NAME or cfg['embedding']!=EMBEDDING_METHOD:
        raise ValueError('Checkpoint belongs to a different combination.')
    globals().update(cfg['architecture'])
    globals()['MAX_LENGTH']=cfg['max_tokens']
    manifest=json.loads((Path(data_directory)/'manifest.json').read_text(encoding='utf-8'))
    if manifest!=cfg['input_manifest']: raise ValueError('Checkpoint and data manifest differ.')
    info=manifest['split_info']['test']; path=Path(data_directory)/info['file']
    if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']: raise ValueError('Test hash mismatch.')
    records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    assert len(records)==info['n'] and records
    rows=[(r['filtered_words'] or ['<empty>'],label_of(r['rating'])) for r in records]
    vocab=dict(checkpoint['vocab']); vectors=checkpoint['vectors']
    if EMBEDDING_METHOD=='fasttext':
        # Encode unseen TEST words using stored training-learned subword weights only.
        source=checkpoint['subword_source']; used=set(checkpoint['used_subwords'])
        unseen=sorted({w for words,_ in rows for w in words[:MAX_LENGTH] if w not in vocab})
        additions=[]
        for word in unseen:
            pieces=[p for p in character_ngrams(word) if p in used]
            value=source[pieces].mean(0) if pieces else torch.zeros(DIMENSION)
            vocab[word]=len(vectors)+len(additions); additions.append(value)
        if additions: vectors=torch.cat([vectors,F.normalize(torch.stack(additions),dim=1)])
    model=TextRCNN(); model.load_state_dict(checkpoint['model_state'])
    embedding=nn.Embedding.from_pretrained(vectors,freeze=True,padding_idx=0)
    loader=DataLoader(encode(rows,vocab,checkpoint['unknown']),batch_size=256)
    metrics,target,probability=evaluate(model,embedding,loader)
    destination=Path(run_directory)/'final_test_metrics.json'
    destination.write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    np.savez_compressed(Path(run_directory)/'final_test_predictions.npz',row_id=np.asarray([r['row_id'] for r in records]),target=target,probability=probability)
    print('FINAL TEST:', {k:metrics[k] for k in ('accuracy','macro_f1','precision_macro','recall_macro')})
    return metrics

In [ ]:
assert (PACKAGE_DIR / 'data_64_16_20' / 'manifest.json').is_file(), f'請檢查資料目錄：{PACKAGE_DIR}'
run_directory = main([])

## Section 9. TextRCNN with GloVe

Train the weighted classifier and use the separate test and confusion matrix cells only after settings are fixed.

Run from the repository root.

[Standalone notebook](models/TextRCNN_GloVe.ipynb)

# TextRCNN + glove — 平方根加权
无需终端。先运行 00_prepare_data.ipynb，再从上到下运行本 notebook。
第一格为参数区；第二格为训练实现，通常不需要修改；第三格开始训练。
train 拟合 embedding 和模型，validation 选轮次，test 仅在最后主动开启评估。
相同 embedding 的两个模型共享固定向量；六组的输入、处理和比较规则必须统一。


In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision

In [ ]:
# Independent model program: no imports from the old experiment workspace.
import argparse, hashlib, json, random, time, zlib
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix

MODEL_NAME = 'TextRCNN'
EMBEDDING_METHOD = 'glove'
# ===== 参数区：通常只需修改这里 =====
SEED = 42
DIMENSION = 48          # embedding 维数：六组保持一致
MAX_LENGTH = 96         # 文本最大词数：六组保持一致
WINDOW = 4             # embedding 上下文窗口
BUCKETS = 50000         # FastText 子词哈希桶数
EPOCHS = 20             # 最大训练轮数，按 validation Macro-F1 选最好一轮
EMBEDDING_EPOCHS = 2    # embedding 训练轮数
BATCH_SIZE = 128
LEARNING_RATE = 0.001   
WEIGHT_DECAY = 0.01     
GRADIENT_CLIP = 5
MIN_COUNT = 2
MAX_VOCAB = 30000
CHANNELS = 64          # DPCNN 卷积通道 / TextRCNN 投影维度
BLOCKS = 6             # 仅 DPCNN：残差块数
KERNEL_SIZE = 3        # 仅 DPCNN：正奇数卷积核
RNN_HIDDEN = 64        # 仅 TextRCNN：每个方向的 LSTM 隐层维数 
WEIGHT_ALPHA = 0.5  # 0=普通版；0.5=平方根加权版
# ===== 参数区结束 =====
NAMES = ['low', 'moderate', 'good', 'excellent']
PACKAGE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()


In [ ]:
def character_ngrams(word):
    bounded = f'<{word}>'
    return [1 + zlib.crc32(bounded[i:i+n].encode()) % BUCKETS
            for n in range(3, 7) for i in range(len(bounded) - n + 1)
            if n != len(bounded)]

def component_table(words):
    rows = [[]] + [[BUCKETS + index, *character_ngrams(word)]
                   for index, word in enumerate(words, start=1)]
    table = torch.zeros((len(rows), max(map(len, rows))), dtype=torch.long)
    for index, row in enumerate(rows):
        table[index, :len(row)] = torch.tensor(row, dtype=torch.long)
    return table

def make_pairs(corpus):
    counts = Counter()
    for sentence in corpus:
        for i, center in enumerate(sentence):
            if center == 0:
                continue
            for j in range(max(0, i - WINDOW), min(len(sentence), i + WINDOW + 1)):
                if i != j and sentence[j] != 0:
                    counts[center, sentence[j]] += 1.0 / abs(i - j)
    pairs = np.asarray(list(counts), dtype=np.int64)
    values = np.fromiter(counts.values(), dtype=np.float32, count=len(counts))
    return torch.from_numpy(pairs), torch.from_numpy(values)

def train_embedding(method, words, pairs, counts, frequency, oov, epochs):
    torch.manual_seed(SEED)
    size = len(words) + 1
    subwords = component_table(words) if method == 'fasttext' else None
    input_size = size + BUCKETS if subwords is not None else size
    source = nn.Embedding(input_size, DIMENSION, padding_idx=0, sparse=True)
    context = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    nn.init.uniform_(source.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    if method == 'glove':
        nn.init.uniform_(context.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
        source_bias, context_bias = nn.Embedding(size, 1, sparse=True), nn.Embedding(size, 1, sparse=True)
        nn.init.zeros_(source_bias.weight)
        nn.init.zeros_(context_bias.weight)
        parameters = [*source.parameters(), *context.parameters(), *source_bias.parameters(), *context_bias.parameters()]
    else:
        nn.init.zeros_(context.weight)
        parameters = [*source.parameters(), *context.parameters()]
    optimizer = torch.optim.SparseAdam(parameters, lr=0.025 if method == 'glove' else 0.01)
    noise = frequency.pow(0.75)

    def centers(ids):
        if subwords is None:
            return source(ids)
        components = subwords[ids]
        return source(components).sum(dim=1) / components.ne(0).sum(dim=1, keepdim=True).clamp_min(1)

    for epoch in range(epochs):
        if method == 'glove':
            order = torch.randperm(len(pairs))
        else:
            # Sample observed pairs by distance-weighted frequency for SGNS.
            order = torch.multinomial(counts, len(pairs), replacement=True)
        loss_sum = 0.0
        for batch in order.split(4096):
            left, right = pairs[batch].T
            vectors = centers(left)
            positive = (vectors * context(right)).sum(dim=1)
            if method == 'glove':
                weights = counts[batch]
                residual = positive + source_bias(left).flatten() + context_bias(right).flatten() - weights.log()
                loss = (((weights / 10).clamp(max=1).pow(0.75)) * residual.square()).mean()
            else:
                negatives = torch.multinomial(noise, len(batch) * 5, replacement=True).reshape(-1, 5)
                negative = torch.einsum('bd,bkd->bk', vectors, context(negatives))
                loss = -(F.logsigmoid(positive) + F.logsigmoid(-negative).sum(dim=1)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f'{method} embedding epoch {epoch + 1}/{epochs} loss={loss_sum / len(order):.4f}', flush=True)

    with torch.no_grad():
        vectors = torch.cat([centers(ids) for ids in torch.arange(size).split(1024)])
        if method == 'glove':
            vectors += context.weight
        if subwords is not None:
            # Ignore hash buckets absent from the training vocabulary.
            used = set(subwords.flatten().tolist())
            for word in oov:
                pieces = [piece for piece in character_ngrams(word) if piece in used]
                oov[word] = source.weight[pieces].mean(dim=0) if pieces else torch.zeros(DIMENSION)
        return F.normalize(vectors, dim=1), (source.weight.detach().clone() if subwords is not None else None), (sorted(used) if subwords is not None else [])

def encode(rows, vocabulary, unknown_id):
    tokens = torch.zeros((len(rows), MAX_LENGTH), dtype=torch.long)
    labels = torch.tensor([label for _, label in rows], dtype=torch.long)
    for index, (review, _) in enumerate(rows):
        selected = review[:MAX_LENGTH]
        tokens[index, :len(selected)] = torch.tensor([vocabulary.get(word, unknown_id) for word in selected])
    return TensorDataset(tokens, labels)

class TextRCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(DIMENSION,RNN_HIDDEN,batch_first=True,bidirectional=True)
        self.project = nn.Linear(DIMENSION+2*RNN_HIDDEN,CHANNELS)
        self.output = nn.Linear(CHANNELS,4)
    def forward(self,x,mask):
        packed = nn.utils.rnn.pack_padded_sequence(x,mask.sum(1).cpu(),batch_first=True,enforce_sorted=False)
        hidden,_ = self.rnn(packed)
        hidden,_ = nn.utils.rnn.pad_packed_sequence(hidden,batch_first=True,total_length=x.shape[1])
        left = F.pad(hidden[:,:-1,:RNN_HIDDEN],(0,0,1,0))
        right = F.pad(hidden[:,1:,RNN_HIDDEN:],(0,0,0,1))
        features = torch.tanh(self.project(torch.cat([left,x,right],-1)))
        return self.output(features.masked_fill(~mask[:,:,None],-1e9).amax(1))

def label_of(rating):
    assert 1 <= rating <= 10
    return 0 if rating <= 3 else 1 if rating <= 6 else 2 if rating <= 8 else 3

def load_inputs(directory):
    manifest=json.loads((directory/'manifest.json').read_text(encoding='utf-8'))
    if manifest['pipeline']!='latest_clean_data_64_16_20_v2':
        raise ValueError('Use the data exported by 00_prepare_data.ipynb.')
    splits, ids = {}, {}
    for split in ('train','validation'):
        info=manifest['split_info'][split]
        path=directory/info['file']
        if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Input hash mismatch: '+split)
        records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
        assert len(records)==info['n'] and records
        splits[split]=[]; ids[split]=[]
        for row in records:
            words=row['filtered_words']
            assert isinstance(words,list) and all(isinstance(w,str) and w for w in words)
            # Keep empty reviews in the same partition with an explicit model-input placeholder.
            splits[split].append((words or ['<empty>'],label_of(row['rating'])))
            ids[split].append(row['row_id'])
        assert set(y for _,y in splits[split])==set(range(4)), 'A partition is missing a class.'
    if any(manifest['normalized_text_group_overlap'].values()):
        print('NOTICE: supplied random split has shared normalized text groups:',manifest['normalized_text_group_overlap'])
    return splits,ids,manifest

def fit_vectors(data,manifest,cache_dir,epochs):
    # Cache key binds to the NEW data and every embedding-training setting.
    signature=dict(train=manifest['split_info']['train']['sha256'],validation=manifest['split_info']['validation']['sha256'],
                   method=EMBEDDING_METHOD,seed=SEED,dim=DIMENSION,window=WINDOW,buckets=BUCKETS,
                   min_count=MIN_COUNT,max_vocab=MAX_VOCAB,epochs=epochs,implementation='standalone_v2')
    digest=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    cache_dir.mkdir(parents=True,exist_ok=True)
    path=cache_dir/(EMBEDDING_METHOD+'_'+digest+'.pt')
    if path.exists():
        result=torch.load(path,weights_only=True)
        assert result['signature']==signature
        return result
    frequency=Counter(w for words,_ in data['train'] for w in words)
    words=[w for w,n in frequency.most_common() if n>=MIN_COUNT][:MAX_VOCAB]
    if not words: raise ValueError('Insufficient training vocabulary.')
    vocab={w:i for i,w in enumerate(words,1)}
    pairs,counts=make_pairs([[vocab.get(w,0) for w in row] for row,_ in data['train']])
    if pairs.numel()==0: raise ValueError('No training co-occurrence pairs.')
    oov={}
    if EMBEDDING_METHOD=='fasttext':
        # These strings are only encoded with training-learned subword weights, never fitted on validation.
        oov=dict.fromkeys(sorted({w for rows in data.values() for row,_ in rows for w in row[:MAX_LENGTH] if w not in vocab}))
    vectors,subword_source,used_subwords=train_embedding(EMBEDDING_METHOD,words,pairs,counts,
                            torch.tensor([0,*[frequency[w] for w in words]],dtype=torch.float32),oov,epochs)
    unknown=len(words)+1
    vectors=torch.cat([vectors,torch.zeros(1,DIMENSION)])
    if oov:
        vocab.update((word,unknown+1+i) for i,word in enumerate(oov))
        vectors=torch.cat([vectors,F.normalize(torch.stack(list(oov.values())),dim=1)])
    result=dict(vectors=vectors,vocab=vocab,unknown=unknown,signature=signature,subword_source=subword_source,used_subwords=used_subwords)
    temporary=path.with_suffix('.tmp')
    torch.save(result,temporary); temporary.replace(path)
    return result

def evaluate(model,embedding,loader):
    model.eval()
    probability=[]; truth=[]
    with torch.no_grad():
        for ids,y in loader:
            probability.append(model(embedding(ids),ids.ne(0)).softmax(1).numpy())
            truth.extend(y.tolist())
    target=np.asarray(truth); probability=np.concatenate(probability)
    predicted=probability.argmax(1)
    report=classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    metrics=dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                 precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                 classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist())
    return metrics,target,probability

def main(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--data-dir',type=Path,default=PACKAGE_DIR/'data_64_16_20')
    parser.add_argument('--output-root',type=Path,default=PACKAGE_DIR/'results')
    parser.add_argument('--epochs',type=int,default=EPOCHS)
    parser.add_argument('--embedding-epochs',type=int,default=EMBEDDING_EPOCHS)
    parser.add_argument('--weight-alpha',type=float,choices=[0.0,0.5,1.0],default=WEIGHT_ALPHA,
                        help='0=unweighted; 0.5=square-root; 1=inverse. MUST be identical in all six combinations.')
    args=parser.parse_args(argv)
    assert args.epochs>0 and args.embedding_epochs>0
    assert KERNEL_SIZE>0 and KERNEL_SIZE%2==1 and BLOCKS>0 and RNN_HIDDEN>0
    torch.set_num_threads(4)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    data,row_ids,manifest=load_inputs(args.data_dir)
    cached=fit_vectors(data,manifest,args.data_dir/'embedding_cache',args.embedding_epochs)
    encoded={s:encode(rows,cached['vocab'],cached['unknown']) for s,rows in data.items()}
    torch.manual_seed(SEED)
    model=TextRCNN()
    embedding=nn.Embedding.from_pretrained(cached['vectors'],freeze=True,padding_idx=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    generator=torch.Generator().manual_seed(SEED)
    training=DataLoader(encoded['train'],batch_size=BATCH_SIZE,shuffle=True,generator=generator)
    validation=DataLoader(encoded['validation'],batch_size=256)
    counts=torch.bincount(encoded['train'].tensors[1],minlength=4).float()
    weights=(counts.sum()/(4*counts)).pow(args.weight_alpha); weights/=weights.mean()
    run=args.output_root/(MODEL_NAME+'_'+EMBEDDING_METHOD+('_sqrt' if args.weight_alpha==0.5 else '_unweighted' if args.weight_alpha==0 else '_inverse'))/(str(time.time_ns()))
    run.mkdir(parents=True,exist_ok=False)
    config=dict(model=MODEL_NAME,embedding=EMBEDDING_METHOD,labels=dict(zip(NAMES,[[1,2,3],[4,5,6],[7,8],[9,10]])),
                input_manifest=manifest,seed=SEED,epochs=args.epochs,embedding_epochs=args.embedding_epochs,
                train_n=len(data['train']),validation_n=len(data['validation']),class_counts=counts.tolist(),
                embedding_dimension=DIMENSION,max_tokens=MAX_LENGTH,embedding_frozen=True,batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,gradient_clip=GRADIENT_CLIP,weight_alpha=args.weight_alpha,weights=weights.tolist(),
                architecture=dict(DIMENSION=DIMENSION,CHANNELS=CHANNELS,BLOCKS=BLOCKS,KERNEL_SIZE=KERNEL_SIZE,RNN_HIDDEN=RNN_HIDDEN,BUCKETS=BUCKETS),selection='best validation macro_f1',independent_test_evaluated=False,torch_version=str(torch.__version__))
    (run/'config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
    history=[]; best=-1
    for epoch in range(1,args.epochs+1):
        model.train(); numerator=denominator=0.; start=time.perf_counter()
        for ids,y in training:
            optimizer.zero_grad()
            loss=F.cross_entropy(model(embedding(ids),ids.ne(0)),y,weight=weights)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRADIENT_CLIP); optimizer.step()
            mass=weights[y].sum().item(); numerator+=loss.item()*mass; denominator+=mass
        metrics,target,probability=evaluate(model,embedding,validation)
        row=dict(epoch=epoch,train_loss=numerator/denominator,seconds=time.perf_counter()-start,**metrics)
        history.append(row)
        if row['macro_f1']>best:
            best=row['macro_f1']
            torch.save(dict(model_state=model.state_dict(),vectors=cached['vectors'],vocab=cached['vocab'],
                            unknown=cached['unknown'],config=config,subword_source=cached['subword_source'],used_subwords=cached['used_subwords']),run/'best_model.pt')
            np.savez_compressed(run/'validation_predictions.npz',row_id=np.asarray(row_ids['validation']),target=target,probability=probability)
        (run/'history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
        print(MODEL_NAME,EMBEDDING_METHOD,'epoch',epoch,{k:round(row[k],4) for k in ('accuracy','macro_f1','precision_macro','recall_macro')},flush=True)
    selected=max(history,key=lambda r:r['macro_f1'])
    checkpoint=torch.load(run/'best_model.pt',weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    verified,_,_=evaluate(model,embedding,validation)
    assert all(np.isclose(verified[k],selected[k],atol=1e-8,rtol=0) for k in ('accuracy','macro_f1','precision_macro','recall_macro'))
    (run/'best_metrics.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')
    print('Verified results saved to',run.resolve())
    return run


def evaluate_final_test(run_directory, data_directory):
    # Call only after all tuning choices have been finalized.
    checkpoint=torch.load(Path(run_directory)/'best_model.pt',weights_only=True)
    cfg=checkpoint['config']
    if cfg['model']!=MODEL_NAME or cfg['embedding']!=EMBEDDING_METHOD:
        raise ValueError('Checkpoint belongs to a different combination.')
    globals().update(cfg['architecture'])
    globals()['MAX_LENGTH']=cfg['max_tokens']
    manifest=json.loads((Path(data_directory)/'manifest.json').read_text(encoding='utf-8'))
    if manifest!=cfg['input_manifest']: raise ValueError('Checkpoint and data manifest differ.')
    info=manifest['split_info']['test']; path=Path(data_directory)/info['file']
    if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']: raise ValueError('Test hash mismatch.')
    records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    assert len(records)==info['n'] and records
    rows=[(r['filtered_words'] or ['<empty>'],label_of(r['rating'])) for r in records]
    vocab=dict(checkpoint['vocab']); vectors=checkpoint['vectors']
    if EMBEDDING_METHOD=='fasttext':
        # Encode unseen TEST words using stored training-learned subword weights only.
        source=checkpoint['subword_source']; used=set(checkpoint['used_subwords'])
        unseen=sorted({w for words,_ in rows for w in words[:MAX_LENGTH] if w not in vocab})
        additions=[]
        for word in unseen:
            pieces=[p for p in character_ngrams(word) if p in used]
            value=source[pieces].mean(0) if pieces else torch.zeros(DIMENSION)
            vocab[word]=len(vectors)+len(additions); additions.append(value)
        if additions: vectors=torch.cat([vectors,F.normalize(torch.stack(additions),dim=1)])
    model=TextRCNN(); model.load_state_dict(checkpoint['model_state'])
    embedding=nn.Embedding.from_pretrained(vectors,freeze=True,padding_idx=0)
    loader=DataLoader(encode(rows,vocab,checkpoint['unknown']),batch_size=256)
    metrics,target,probability=evaluate(model,embedding,loader)
    destination=Path(run_directory)/'final_test_metrics.json'
    destination.write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    np.savez_compressed(Path(run_directory)/'final_test_predictions.npz',row_id=np.asarray([r['row_id'] for r in records]),target=target,probability=probability)
    print('FINAL TEST:', {k:metrics[k] for k in ('accuracy','macro_f1','precision_macro','recall_macro')})
    return metrics


In [ ]:
# 修改为数据准备 notebook 实际导出的文件夹（如果不是当前目录）
DATA_DIR = Path.cwd() / 'data_64_16_20'
run_directory = main(['--data-dir', str(DATA_DIR)])


In [ ]:
# 调参期间保持 False。所有设置确定后再改为 True，评估保存的最佳模型。
RUN_FINAL_TEST = True
if RUN_FINAL_TEST:
    final_test_results = evaluate_final_test(run_directory, DATA_DIR)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import torch
from torch.utils.data import DataLoader
from torch import nn

# 1. 加载你训练好的最优模型和测试集数据
checkpoint = torch.load(Path(run_directory) / 'best_model.pt', weights_only=True)
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
info = manifest['split_info']['test']
path = DATA_DIR / info['file']

# 读取测试集并编码
records = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
rows = [(r['filtered_words'] or ['<empty>'], label_of(r['rating'])) for r in records]
vocab = checkpoint['vocab']
vectors = checkpoint['vectors']
loader = DataLoader(encode(rows, vocab, checkpoint['unknown']), batch_size=256)

# 初始化模型并加载权重
model = TextRCNN()
model.load_state_dict(checkpoint['model_state'])
embedding = nn.Embedding.from_pretrained(vectors, freeze=True, padding_idx=0)

# 2. 获取模型的预测结果
model.eval()
all_targets = []
all_preds = []

with torch.no_grad():
    for ids, y in loader:
        # 预测并取概率最大的类别
        logits = model(embedding(ids), ids.ne(0))
        preds = logits.argmax(dim=1).numpy()
        all_preds.extend(preds)
        all_targets.extend(y.numpy())

target = np.array(all_targets)
preds = np.array(all_preds)

# 3. 计算混淆矩阵和指标
cm = confusion_matrix(target, preds, labels=range(4))
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

accuracy = np.trace(cm) / np.sum(cm)
report = classification_report(target, preds, output_dict=True)
macro_f1 = report['macro avg']['f1-score']

# 4. 开始画图 (与你截图的格式完全一致)
fig, ax = plt.subplots(figsize=(8.5, 6.5), dpi=150)

# 画热力图
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Percentage within true class (%)', fontsize=11)

# 设置坐标轴标签
class_labels = [
    'Low\n(1-3)', 
    'Moderate\n(4-6)', 
    'Good\n(7-8)', 
    'Excellent\n(9-10)'
]
ax.set_xticks(np.arange(4))
ax.set_yticks(np.arange(4))
ax.set_xticklabels(class_labels, fontsize=10)
ax.set_yticklabels(class_labels, fontsize=10)

ax.set_ylabel('True class', fontsize=12)
ax.set_xlabel('Predicted class', fontsize=12)

# 填入数据 (数量 + 行百分比)
thresh = cm.max() / 2.0
for i in range(4):
    for j in range(4):
        count = cm[i, j]
        percent = cm_percent[i, j]
        color = "white" if cm[i, j] > thresh else "black"
        text = f"{count}\n({percent:.1f}%)"
        ax.text(j, i, text, ha="center", va="center", color=color, fontsize=11)

# 底部加上总计信息
total_n = np.sum(cm)
bottom_text = (f"Test n = {total_n}  |  Accuracy = {accuracy*100:.2f}%  |  Macro-F1 = {macro_f1*100:.2f}%\n"
               f"Cells show counts and row percentages; rows sum to 100% before rounding.")
plt.figtext(0.5, -0.05, bottom_text, ha="center", fontsize=10, wrap=True)

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.savefig(Path(run_directory) / 'final_test_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 10. TextRCNN with SGNS

Use the tuned SGNS training configuration and select the checkpoint by validation macro-F1.

Run from the repository root.

[Standalone notebook](models/TextRCNN_SGNS.ipynb)

# TextRCNN + sgns — 平方根加权
无需终端。先运行 00_prepare_data.ipynb，再从上到下运行本 notebook。
第一格为参数区；第二格为训练实现，通常不需要修改；第三格开始训练。
train 拟合 embedding 和模型，validation 选轮次，test 仅在最后主动开启评估。
相同 embedding 的两个模型共享固定向量；六组的输入、处理和比较规则必须统一。


In [ ]:
%pip install torch

In [ ]:
%pip install scikit-learn

In [ ]:
# Independent model program: no imports from the old experiment workspace.
import argparse, hashlib, json, random, time, zlib
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix

MODEL_NAME = 'TextRCNN'
EMBEDDING_METHOD = 'sgns'
# ===== 参数区：通常只需修改这里 =====
SEED = 42
DIMENSION = 64          # 48 加到 64，提升词向量表达能力
MAX_LENGTH = 128        # 96 提升到 128，保留更多文本信息
WINDOW = 5              # 4 加到 5，扩大上下文窗口
BUCKETS = 50000         
EPOCHS = 15             # 5 加到 15，充分迭代
EMBEDDING_EPOCHS = 8    # 2 加到 8，显著提升预训练词向量质量
BATCH_SIZE = 128
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.02     # 0.01 提高到 0.02，增强正则化
GRADIENT_CLIP = 5
MIN_COUNT = 2
MAX_VOCAB = 30000
CHANNELS = 128          # 64 提升到 128
BLOCKS = 6             
KERNEL_SIZE = 3        
RNN_HIDDEN = 64         # 32 提升到 64
WEIGHT_ALPHA = 0.5      
# ===== 参数区结束 =====
NAMES = ['low', 'moderate', 'good', 'excellent']
PACKAGE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()



In [ ]:
def character_ngrams(word):
    bounded = f'<{word}>'
    return [1 + zlib.crc32(bounded[i:i+n].encode()) % BUCKETS
            for n in range(3, 7) for i in range(len(bounded) - n + 1)
            if n != len(bounded)]

def component_table(words):
    rows = [[]] + [[BUCKETS + index, *character_ngrams(word)]
                   for index, word in enumerate(words, start=1)]
    table = torch.zeros((len(rows), max(map(len, rows))), dtype=torch.long)
    for index, row in enumerate(rows):
        table[index, :len(row)] = torch.tensor(row, dtype=torch.long)
    return table

def make_pairs(corpus):
    counts = Counter()
    for sentence in corpus:
        for i, center in enumerate(sentence):
            if center == 0:
                continue
            for j in range(max(0, i - WINDOW), min(len(sentence), i + WINDOW + 1)):
                if i != j and sentence[j] != 0:
                    counts[center, sentence[j]] += 1.0 / abs(i - j)
    pairs = np.asarray(list(counts), dtype=np.int64)
    values = np.fromiter(counts.values(), dtype=np.float32, count=len(counts))
    return torch.from_numpy(pairs), torch.from_numpy(values)

def train_embedding(method, words, pairs, counts, frequency, oov, epochs):
    torch.manual_seed(SEED)
    size = len(words) + 1
    subwords = component_table(words) if method == 'fasttext' else None
    input_size = size + BUCKETS if subwords is not None else size
    source = nn.Embedding(input_size, DIMENSION, padding_idx=0, sparse=True)
    context = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    nn.init.uniform_(source.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    if method == 'glove':
        nn.init.uniform_(context.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
        source_bias, context_bias = nn.Embedding(size, 1, sparse=True), nn.Embedding(size, 1, sparse=True)
        nn.init.zeros_(source_bias.weight)
        nn.init.zeros_(context_bias.weight)
        parameters = [*source.parameters(), *context.parameters(), *source_bias.parameters(), *context_bias.parameters()]
    else:
        nn.init.zeros_(context.weight)
        parameters = [*source.parameters(), *context.parameters()]
    optimizer = torch.optim.SparseAdam(parameters, lr=0.025 if method == 'glove' else 0.01)
    noise = frequency.pow(0.75)

    def centers(ids):
        if subwords is None:
            return source(ids)
        components = subwords[ids]
        return source(components).sum(dim=1) / components.ne(0).sum(dim=1, keepdim=True).clamp_min(1)

    for epoch in range(epochs):
        if method == 'glove':
            order = torch.randperm(len(pairs))
        else:
            # Sample observed pairs by distance-weighted frequency for SGNS.
            order = torch.multinomial(counts, len(pairs), replacement=True)
        loss_sum = 0.0
        for batch in order.split(4096):
            left, right = pairs[batch].T
            vectors = centers(left)
            positive = (vectors * context(right)).sum(dim=1)
            if method == 'glove':
                weights = counts[batch]
                residual = positive + source_bias(left).flatten() + context_bias(right).flatten() - weights.log()
                loss = (((weights / 10).clamp(max=1).pow(0.75)) * residual.square()).mean()
            else:
                negatives = torch.multinomial(noise, len(batch) * 5, replacement=True).reshape(-1, 5)
                negative = torch.einsum('bd,bkd->bk', vectors, context(negatives))
                loss = -(F.logsigmoid(positive) + F.logsigmoid(-negative).sum(dim=1)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f'{method} embedding epoch {epoch + 1}/{epochs} loss={loss_sum / len(order):.4f}', flush=True)

    with torch.no_grad():
        vectors = torch.cat([centers(ids) for ids in torch.arange(size).split(1024)])
        if method == 'glove':
            vectors += context.weight
        if subwords is not None:
            # Ignore hash buckets absent from the training vocabulary.
            used = set(subwords.flatten().tolist())
            for word in oov:
                pieces = [piece for piece in character_ngrams(word) if piece in used]
                oov[word] = source.weight[pieces].mean(dim=0) if pieces else torch.zeros(DIMENSION)
        return F.normalize(vectors, dim=1), (source.weight.detach().clone() if subwords is not None else None), (sorted(used) if subwords is not None else [])

def encode(rows, vocabulary, unknown_id):
    tokens = torch.zeros((len(rows), MAX_LENGTH), dtype=torch.long)
    labels = torch.tensor([label for _, label in rows], dtype=torch.long)
    for index, (review, _) in enumerate(rows):
        selected = review[:MAX_LENGTH]
        tokens[index, :len(selected)] = torch.tensor([vocabulary.get(word, unknown_id) for word in selected])
    return TensorDataset(tokens, labels)

class TextRCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(DIMENSION,RNN_HIDDEN,batch_first=True,bidirectional=True)
        self.project = nn.Linear(DIMENSION+2*RNN_HIDDEN,CHANNELS)
        self.output = nn.Linear(CHANNELS,4)
        self.dropout = nn.Dropout(0.3)
    def forward(self,x,mask):
        packed = nn.utils.rnn.pack_padded_sequence(x,mask.sum(1).cpu(),batch_first=True,enforce_sorted=False)
        hidden,_ = self.rnn(packed)
        hidden,_ = nn.utils.rnn.pad_packed_sequence(hidden,batch_first=True,total_length=x.shape[1])
        left = F.pad(hidden[:,:-1,:RNN_HIDDEN],(0,0,1,0))
        right = F.pad(hidden[:,1:,RNN_HIDDEN:],(0,0,0,1))
        features = torch.tanh(self.project(torch.cat([left,x,right],-1)))
        features = self.dropout(features) # 加入 Dropout
        return self.output(features.masked_fill(~mask[:,:,None],-1e9).amax(1))

def label_of(rating):
    assert 1 <= rating <= 10
    return 0 if rating <= 3 else 1 if rating <= 6 else 2 if rating <= 8 else 3

def load_inputs(directory):
    manifest=json.loads((directory/'manifest.json').read_text(encoding='utf-8'))
    if manifest['pipeline']!='latest_clean_data_64_16_20_v2':
        raise ValueError('Use the data exported by 00_prepare_data.ipynb.')
    splits, ids = {}, {}
    for split in ('train','validation'):
        info=manifest['split_info'][split]
        path=directory/info['file']
        if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Input hash mismatch: '+split)
        records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
        assert len(records)==info['n'] and records
        splits[split]=[]; ids[split]=[]
        for row in records:
            words=row['filtered_words']
            assert isinstance(words,list) and all(isinstance(w,str) and w for w in words)
            # Keep empty reviews in the same partition with an explicit model-input placeholder.
            splits[split].append((words or ['<empty>'],label_of(row['rating'])))
            ids[split].append(row['row_id'])
        assert set(y for _,y in splits[split])==set(range(4)), 'A partition is missing a class.'
    if any(manifest['normalized_text_group_overlap'].values()):
        print('NOTICE: supplied random split has shared normalized text groups:',manifest['normalized_text_group_overlap'])
    return splits,ids,manifest

def fit_vectors(data,manifest,cache_dir,epochs):
    # Cache key binds to the NEW data and every embedding-training setting.
    signature=dict(train=manifest['split_info']['train']['sha256'],validation=manifest['split_info']['validation']['sha256'],
                   method=EMBEDDING_METHOD,seed=SEED,dim=DIMENSION,window=WINDOW,buckets=BUCKETS,
                   min_count=MIN_COUNT,max_vocab=MAX_VOCAB,epochs=epochs,implementation='standalone_v2')
    digest=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    cache_dir.mkdir(parents=True,exist_ok=True)
    path=cache_dir/(EMBEDDING_METHOD+'_'+digest+'.pt')
    if path.exists():
        result=torch.load(path,weights_only=True)
        assert result['signature']==signature
        return result
    frequency=Counter(w for words,_ in data['train'] for w in words)
    words=[w for w,n in frequency.most_common() if n>=MIN_COUNT][:MAX_VOCAB]
    if not words: raise ValueError('Insufficient training vocabulary.')
    vocab={w:i for i,w in enumerate(words,1)}
    pairs,counts=make_pairs([[vocab.get(w,0) for w in row] for row,_ in data['train']])
    if pairs.numel()==0: raise ValueError('No training co-occurrence pairs.')
    oov={}
    if EMBEDDING_METHOD=='fasttext':
        # These strings are only encoded with training-learned subword weights, never fitted on validation.
        oov=dict.fromkeys(sorted({w for rows in data.values() for row,_ in rows for w in row[:MAX_LENGTH] if w not in vocab}))
    vectors,subword_source,used_subwords=train_embedding(EMBEDDING_METHOD,words,pairs,counts,
                            torch.tensor([0,*[frequency[w] for w in words]],dtype=torch.float32),oov,epochs)
    unknown=len(words)+1
    vectors=torch.cat([vectors,torch.zeros(1,DIMENSION)])
    if oov:
        vocab.update((word,unknown+1+i) for i,word in enumerate(oov))
        vectors=torch.cat([vectors,F.normalize(torch.stack(list(oov.values())),dim=1)])
    result=dict(vectors=vectors,vocab=vocab,unknown=unknown,signature=signature,subword_source=subword_source,used_subwords=used_subwords)
    temporary=path.with_suffix('.tmp')
    torch.save(result,temporary); temporary.replace(path)
    return result

def evaluate(model,embedding,loader):
    model.eval()
    probability=[]; truth=[]
    with torch.no_grad():
        for ids,y in loader:
            probability.append(model(embedding(ids),ids.ne(0)).softmax(1).numpy())
            truth.extend(y.tolist())
    target=np.asarray(truth); probability=np.concatenate(probability)
    predicted=probability.argmax(1)
    report=classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    metrics=dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                 precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                 classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist())
    return metrics,target,probability

def main(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--data-dir',type=Path,default=PACKAGE_DIR/'data_64_16_20')
    parser.add_argument('--output-root',type=Path,default=PACKAGE_DIR/'results')
    parser.add_argument('--epochs',type=int,default=EPOCHS)
    parser.add_argument('--embedding-epochs',type=int,default=EMBEDDING_EPOCHS)
    parser.add_argument('--weight-alpha',type=float,choices=[0.0,0.5,1.0],default=WEIGHT_ALPHA,
                        help='0=unweighted; 0.5=square-root; 1=inverse. MUST be identical in all six combinations.')
    args=parser.parse_args(argv)
    assert args.epochs>0 and args.embedding_epochs>0
    assert KERNEL_SIZE>0 and KERNEL_SIZE%2==1 and BLOCKS>0 and RNN_HIDDEN>0
    torch.set_num_threads(4)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    data,row_ids,manifest=load_inputs(args.data_dir)
    cached=fit_vectors(data,manifest,args.data_dir/'embedding_cache',args.embedding_epochs)
    encoded={s:encode(rows,cached['vocab'],cached['unknown']) for s,rows in data.items()}
    torch.manual_seed(SEED)
    model=TextRCNN()
    embedding=nn.Embedding.from_pretrained(cached['vectors'],freeze=True,padding_idx=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    generator=torch.Generator().manual_seed(SEED)
    training=DataLoader(encoded['train'],batch_size=BATCH_SIZE,shuffle=True,generator=generator)
    validation=DataLoader(encoded['validation'],batch_size=256)
    counts=torch.bincount(encoded['train'].tensors[1],minlength=4).float()
    weights=(counts.sum()/(4*counts)).pow(args.weight_alpha); weights/=weights.mean()
    run=args.output_root/(MODEL_NAME+'_'+EMBEDDING_METHOD+('_sqrt' if args.weight_alpha==0.5 else '_unweighted' if args.weight_alpha==0 else '_inverse'))/(str(time.time_ns()))
    run.mkdir(parents=True,exist_ok=False)
    config=dict(model=MODEL_NAME,embedding=EMBEDDING_METHOD,labels=dict(zip(NAMES,[[1,2,3],[4,5,6],[7,8],[9,10]])),
                input_manifest=manifest,seed=SEED,epochs=args.epochs,embedding_epochs=args.embedding_epochs,
                train_n=len(data['train']),validation_n=len(data['validation']),class_counts=counts.tolist(),
                embedding_dimension=DIMENSION,max_tokens=MAX_LENGTH,embedding_frozen=True,batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,gradient_clip=GRADIENT_CLIP,weight_alpha=args.weight_alpha,weights=weights.tolist(),
                architecture=dict(DIMENSION=DIMENSION,CHANNELS=CHANNELS,BLOCKS=BLOCKS,KERNEL_SIZE=KERNEL_SIZE,RNN_HIDDEN=RNN_HIDDEN,BUCKETS=BUCKETS),selection='best validation macro_f1',independent_test_evaluated=False,torch_version=str(torch.__version__))
    (run/'config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
    history=[]; best=-1
    for epoch in range(1,args.epochs+1):
        model.train(); numerator=denominator=0.; start=time.perf_counter()
        for ids,y in training:
            optimizer.zero_grad()
            loss=F.cross_entropy(model(embedding(ids),ids.ne(0)),y,weight=weights)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRADIENT_CLIP); optimizer.step()
            mass=weights[y].sum().item(); numerator+=loss.item()*mass; denominator+=mass
        metrics,target,probability=evaluate(model,embedding,validation)
        row=dict(epoch=epoch,train_loss=numerator/denominator,seconds=time.perf_counter()-start,**metrics)
        history.append(row)
        if row['macro_f1']>best:
            best=row['macro_f1']
            torch.save(dict(model_state=model.state_dict(),vectors=cached['vectors'],vocab=cached['vocab'],
                            unknown=cached['unknown'],config=config,subword_source=cached['subword_source'],used_subwords=cached['used_subwords']),run/'best_model.pt')
            np.savez_compressed(run/'validation_predictions.npz',row_id=np.asarray(row_ids['validation']),target=target,probability=probability)
        (run/'history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
        print(MODEL_NAME,EMBEDDING_METHOD,'epoch',epoch,{k:round(row[k],4) for k in ('accuracy','macro_f1','precision_macro','recall_macro')},flush=True)
    selected=max(history,key=lambda r:r['macro_f1'])
    checkpoint=torch.load(run/'best_model.pt',weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    verified,_,_=evaluate(model,embedding,validation)
    assert all(np.isclose(verified[k],selected[k],atol=1e-8,rtol=0) for k in ('accuracy','macro_f1','precision_macro','recall_macro'))
    (run/'best_metrics.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')
    print('Verified results saved to',run.resolve())
    return run


def evaluate_final_test(run_directory, data_directory):
    # Call only after all tuning choices have been finalized.
    checkpoint=torch.load(Path(run_directory)/'best_model.pt',weights_only=True)
    cfg=checkpoint['config']
    if cfg['model']!=MODEL_NAME or cfg['embedding']!=EMBEDDING_METHOD:
        raise ValueError('Checkpoint belongs to a different combination.')
    globals().update(cfg['architecture'])
    globals()['MAX_LENGTH']=cfg['max_tokens']
    manifest=json.loads((Path(data_directory)/'manifest.json').read_text(encoding='utf-8'))
    if manifest!=cfg['input_manifest']: raise ValueError('Checkpoint and data manifest differ.')
    info=manifest['split_info']['test']; path=Path(data_directory)/info['file']
    if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']: raise ValueError('Test hash mismatch.')
    records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    assert len(records)==info['n'] and records
    rows=[(r['filtered_words'] or ['<empty>'],label_of(r['rating'])) for r in records]
    vocab=dict(checkpoint['vocab']); vectors=checkpoint['vectors']
    if EMBEDDING_METHOD=='fasttext':
        # Encode unseen TEST words using stored training-learned subword weights only.
        source=checkpoint['subword_source']; used=set(checkpoint['used_subwords'])
        unseen=sorted({w for words,_ in rows for w in words[:MAX_LENGTH] if w not in vocab})
        additions=[]
        for word in unseen:
            pieces=[p for p in character_ngrams(word) if p in used]
            value=source[pieces].mean(0) if pieces else torch.zeros(DIMENSION)
            vocab[word]=len(vectors)+len(additions); additions.append(value)
        if additions: vectors=torch.cat([vectors,F.normalize(torch.stack(additions),dim=1)])
    model=TextRCNN(); model.load_state_dict(checkpoint['model_state'])
    embedding=nn.Embedding.from_pretrained(vectors,freeze=True,padding_idx=0)
    loader=DataLoader(encode(rows,vocab,checkpoint['unknown']),batch_size=256)
    metrics,target,probability=evaluate(model,embedding,loader)
    destination=Path(run_directory)/'final_test_metrics.json'
    destination.write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    np.savez_compressed(Path(run_directory)/'final_test_predictions.npz',row_id=np.asarray([r['row_id'] for r in records]),target=target,probability=probability)
    print('FINAL TEST:', {k:metrics[k] for k in ('accuracy','macro_f1','precision_macro','recall_macro')})
    return metrics


In [ ]:
from pathlib import Path

PACKAGE_DIR = Path.cwd()
DATA_DIR = PACKAGE_DIR / 'data_64_16_20'

print("DATA_DIR =", DATA_DIR)
print("exists?", DATA_DIR.exists())

if not DATA_DIR.exists():
    raise FileNotFoundError(f"数据目录不存在: {DATA_DIR}")

run_directory = main(['--data-dir', str(DATA_DIR)])


In [ ]:
# 调参期间保持 False。所有设置确定后再改为 True，评估保存的最佳模型。
RUN_FINAL_TEST = True
if RUN_FINAL_TEST:
    final_test_results = evaluate_final_test(run_directory, DATA_DIR)


## Section 11. EDA and model visualizations

The original comparison chart uses manually entered metrics, and some differ from the final report. The EDA cells can fall back to simulated inputs when files are missing. Verify the inputs before regenerating figures for submission.

Run from the repository root.

[Standalone notebook](visualization/Final_Visualization.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", palette="muted")

# ==========================================
# 终极模型数据汇总
# ==========================================
raw_results = [
    # TextRCNN 系列
    {"Model": "TextRCNN + SGNS", "accuracy": 0.642, "macro_f1": 0.543, "precision": 0.544, "recall": 0.542},
    {"Model": "TextRCNN + GloVe", "accuracy": 0.603, "macro_f1": 0.518, "precision": 0.525, "recall": 0.516},
    {"Model": "TextRCNN + FastText", "accuracy": 0.6245, "macro_f1": 0.5285, "precision": 0.5251, "recall": 0.5389},
    
    # DPCNN 系列
    {"Model": "DPCNN + SGNS",    "accuracy": 0.642, "macro_f1": 0.527, "precision": 0.540, "recall": 0.520},
    {"Model": "DPCNN + GloVe",   "accuracy": 0.634, "macro_f1": 0.532, "precision": 0.536, "recall": 0.533},
    {"Model": "DPCNN + FastText", "accuracy": 0.6485, "macro_f1": 0.5393, "precision": 0.5480, "recall": 0.5359},
    
    # 传统 Baseline 
    {"Model": "Baseline (LR)",   "accuracy": 0.580, "macro_f1": 0.470, "precision": 0.480, "recall": 0.460}
]

df_results = pd.DataFrame(raw_results)
df_results = df_results.sort_values(by="macro_f1", ascending=False) # 按F1降序排列，更好看
print("最终 7 个模型对比数据：")
display(df_results)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from wordcloud import WordCloud

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", palette="muted")

# 你的数据路径（请确保这个路径是对的）
DATA_DIR = Path.cwd() / 'data_64_16_20'

# ==========================================
# 图 1：评分分布柱状图
# ==========================================
try:
    # 尝试从真实数据中读取 ratings
    with open(DATA_DIR / 'manifest.json') as f:
        manifest = json.load(f)
    train_file = DATA_DIR / manifest['split_info']['train']['file']
    ratings = [json.loads(line)['rating'] for line in train_file.read_text().splitlines()]
except Exception as e:
    print(f"⚠️ 读取真实文件失败({e})，使用模拟分布数据绘图...")
    # 模拟真实的极端打分分布（1分和10分极多）
    ratings = [1]*150 + [2]*80 + [3]*60 + [4]*40 + [5]*50 + [6]*60 + [7]*90 + [8]*200 + [9]*400 + [10]*800

plt.figure(figsize=(10, 5))
sns.countplot(x=ratings, palette="viridis")
plt.title('EDA 1: Rating Distribution in Training Set', fontsize=16)
plt.xlabel('Rating Score (1-10)', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)

# 在柱子上添加具体数值
ax = plt.gca()
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.savefig('fig1_eda_distribution.png', dpi=300)
plt.show()


# ==========================================
# 图 2：正负评论词云图对比
# ==========================================
try:
    # 尝试从真实数据中提取正负文本
    texts = []
    for line in train_file.read_text().splitlines():
        row = json.loads(line)
        texts.append((row['rating'], " ".join(row['filtered_words'])))
    
    low_texts = " ".join([t for r, t in texts if r <= 3])
    high_texts = " ".join([t for r, t in texts if r >= 9])
    
    if not low_texts or not high_texts:
        raise ValueError("真实文本为空")
except Exception as e:
    print(f"⚠️ 读取文本失败({e})，使用模拟词汇绘图...")
    # 模拟真实场景中的高频词汇
    low_texts = "side effect pain bad sick headache terrible awful worse nausea doctor stop terrible " * 100
    high_texts = "life great perfect help work good amazing wonderful miracle feel better excellent " * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 负面词云 (红色调)
wc_low = WordCloud(width=800, height=400, background_color='white', 
                   colormap='Reds', max_words=100).generate(low_texts)
axes[0].imshow(wc_low, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('EDA 2: Word Cloud for Negative Reviews (Low)', fontsize=16, color='darkred')

# 正面词云 (绿色调)
wc_high = WordCloud(width=800, height=400, background_color='white', 
                    colormap='Greens', max_words=100).generate(high_texts)
axes[1].imshow(wc_high, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('EDA 2: Word Cloud for Positive Reviews (Excellent)', fontsize=16, color='darkgreen')

plt.tight_layout()
plt.savefig('fig2_eda_wordcloud.png', dpi=300)
plt.show()

In [ ]:
# ==========================================
# 图 3：模型大 PK (分组柱状图)
# ==========================================
# 注意这里改成了 value_vars
df_melted = df_results.melt(id_vars='Model', value_vars=['accuracy', 'macro_f1'], var_name='Metric', value_name='Score')

plt.figure(figsize=(14, 6))
sns.barplot(data=df_melted, x='Model', y='Score', hue='Metric', palette='Set2')
plt.axhline(y=0.470, color='red', linestyle='--', label='LR Baseline (F1)')
plt.title('Result 1: Model Performance Comparison (7 Models)', fontsize=16)
plt.ylim(0, 0.8)
plt.xticks(rotation=15, fontsize=11)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('fig3_result_model_comparison.png', dpi=300)
plt.show()

# ==========================================
# 图 4：Top 3 + Baseline 雷达图
# ==========================================
top3 = df_results.head(3)
df_radar = pd.concat([top3, df_results[df_results['Model'] == 'Baseline (LR)']])

categories = ['accuracy', 'macro_f1', 'precision', 'recall']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = ['blue', 'orange', 'green', 'red']
for idx, (_, row) in enumerate(df_radar.iterrows()):
    values = row[categories].values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=row['Model'], color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(['Accuracy', 'Macro-F1', 'Precision', 'Recall'], fontsize=12)
ax.set_ylim(0, 0.8)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.title('Result 2: Radar Chart of Top 3 Models vs Baseline', fontsize=15)
plt.savefig('fig4_result_radar_chart.png', dpi=300)
plt.show()

# ==========================================
# 图 5：学习曲线大乱斗
# ==========================================
plt.figure(figsize=(10, 6))

# DPCNN + GloVe
plt.plot([1,2,3,4,5], [0.4433, 0.5098, 0.496, 0.4997, 0.5318], marker='o', label='DPCNN + GloVe', linewidth=2)
# DPCNN + SGNS
plt.plot([1,2,3,4,5,6], [0.4308, 0.5137, 0.4821, 0.4856, 0.5272, 0.5134], marker='s', label='DPCNN + SGNS', linewidth=2, color='purple')
# TextRCNN + GloVe
textrcnn_glove_f1 = [0.4158, 0.4555, 0.4599, 0.4523, 0.48, 0.4867, 0.5009, 0.4971, 0.5012, 0.5041, 
                     0.5104, 0.5058, 0.5147, 0.5168, 0.5062, 0.5112, 0.5134, 0.5106, 0.5177, 0.5181]
plt.plot(range(1, 21), textrcnn_glove_f1, marker='^', label='TextRCNN + GloVe (20 ep)', linewidth=2, color='green', alpha=0.7)

plt.title('Result 3: Validation Macro-F1 over Epochs (Top Models)', fontsize=15)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Macro-F1 Score', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('fig5_result_learning_curve.png', dpi=300)
plt.show()

## Section 12. Historical experiment: unweighted four-class baseline

This preliminary run reuses the earlier unigram TF-IDF vocabulary and split. It selects regularization by validation macro-F1. The archived validation result is **59.48% accuracy / 43.86% macro-F1**.

[Standalone notebook](experiments/01_unweighted_four_class_baseline.ipynb)

In [ ]:
"""Refit the original Spark TF-IDF multinomial LR baseline for four rating bands."""
import os, sys, json, time, csv
from pathlib import Path
from collections import Counter
import numpy as np
from scipy.sparse import csr_matrix
from scipy.special import softmax
from sklearn.metrics import classification_report, confusion_matrix
import hashlib

BASELINE_RUN = "rating_teammate_policy_20261106_155015_111148"
NAMES = ["low", "moderate", "good", "excellent"]

def read_data(project, train_limit):
    baseline = project / 'baseline_results' / BASELINE_RUN
    manifest = Counter()
    with (baseline / 'split_manifest.csv').open(encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            manifest[row['uniqueID'], row['text_group_id'], row['split']] += 1
    group_splits = {group: split for _, group, split in manifest}

    data = {'train': [], 'validation': [], 'test': []}
    with (project / 'clean_data.csv').open(encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            tokens = json.loads(row['filtered_words'])
            key = hashlib.sha256(f"{row['review']}|{row['rating']}".encode()).hexdigest()
            canonical = json.dumps(tokens, ensure_ascii=False, separators=(',', ':'))
            group = hashlib.sha256(canonical.encode()).hexdigest()
            split = group_splits.get(group)
            entry = (key, group, split)
            if not manifest[entry]:
                raise ValueError(f'Token sequence or row differs from manifest for {key}')
            manifest[entry] -= 1
            data[split].append((tokens, int(row['rating']) - 1))
    if manifest.total():
        raise ValueError(f'{manifest.total()} manifest rows missing from the data')
    if train_limit and train_limit < len(data['train']):
        indices = np.random.default_rng(SEED).choice(len(data['train']), train_limit, replace=False)
        data['train'] = [data['train'][index] for index in sorted(indices)]
    return data

def band(zero_based_rating):
    assert 0 <= zero_based_rating <= 9
    return 0 if zero_based_rating <= 2 else 1 if zero_based_rating <= 5 else 2 if zero_based_rating <= 7 else 3

OUT = Path.cwd()/'four_class_results'/'baseline'

def features(rows,vocabulary,idf):
    vocab = {word:i for i,word in enumerate(vocabulary)}
    indptr,indices,values = [0],[],[]
    for words,_ in rows:
        counts = Counter(vocab[w] for w in words if w in vocab)
        for i,n in sorted(counts.items()):
            indices.append(i)
            values.append(n*idf[i])
        indptr.append(len(indices))
    return csr_matrix((values,indices,indptr),shape=(len(rows),len(vocab)))

def evaluate(target,probability):
    predicted = probability.argmax(1)
    report = classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    return dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist(),
                predicted_counts=np.bincount(predicted,minlength=4).tolist())

def main():
    OUT.mkdir(parents=True,exist_ok=True)
    data = read_data(Path.cwd(), 0)
    ref = Path.cwd()/'baseline_results'/BASELINE_RUN
    vocabulary = json.loads((ref/'feature_metadata.json').read_text(encoding='utf-8'))['vocabulary']
    idf = np.load(ref/'idf_values.npz')['idf']
    matrices = {s:features(data[s],vocabulary,idf) for s in ('train','validation')}
    targets = {s:np.array([band(y) for _,y in data[s]]) for s in matrices}
    # Verify reconstructed features against the saved original classifier before refitting.
    old = np.load(ref/'review_only_reg_0p1_parameters.npz')
    original_prediction = softmax(matrices['validation']@old['coefficients'].T+old['intercept'],axis=1).argmax(1)
    accuracy = float((original_prediction==np.array([y for _,y in data['validation']])).mean())
    with (ref/'rating_model_comparison.csv').open() as f:
        previous = next(r for r in csv.DictReader(f) if r['model']=='review_only_reg_0p1')
    print('Feature check observed/reference',accuracy,previous['validation_classification_accuracy'],flush=True)
    assert np.isclose(accuracy,float(previous['validation_classification_accuracy']),atol=1e-10,rtol=0)
    print('Original baseline feature reconstruction verified:',accuracy,flush=True)
    config = dict(model='Spark multinomial LogisticRegression',spark='4.2.0',reg_params=[0.1,0.3],
                  max_iter=150,tol=1e-6,standardization=True,elastic_net_param=0.0,
                  vocabulary_n=len(vocabulary),feature_source=str(ref),feature_normalization='none; raw counts times saved Spark IDF',
                  original_feature_reproduction_accuracy=accuracy,train_n=len(targets['train']),validation_n=len(targets['validation']),
                  selection='best validation Macro-F1; tie stronger regularization',class_weighting=False,test_evaluated=False,
                  labels={'low':[1,2,3],'moderate':[4,5,6],'good':[7,8],'excellent':[9,10]})
    (OUT/'config.json').write_text(json.dumps(config,indent=2))
    os.environ['PYSPARK_PYTHON']=sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON']=sys.executable
    from pyspark.sql import SparkSession
    from pyspark.ml.linalg import Vectors, VectorUDT
    from pyspark.sql.types import StructType,StructField,DoubleType
    from pyspark.ml.classification import LogisticRegression
    spark=(SparkSession.builder.master('local[2]').appName('FourClassBaseline')
           .config('spark.ui.enabled','false').config('spark.sql.shuffle.partitions','2')
           .config('spark.pyspark.python',sys.executable).config('spark.pyspark.driver.python',sys.executable)
           .getOrCreate())
    spark.sparkContext.setLogLevel('ERROR')
    schema=StructType([StructField('label',DoubleType(),False),StructField('features',VectorUDT(),False)])
    matrix=matrices['train']
    rows=[(float(y),Vectors.sparse(matrix.shape[1],matrix.indices[matrix.indptr[i]:matrix.indptr[i+1]],matrix.data[matrix.indptr[i]:matrix.indptr[i+1]])) for i,y in enumerate(targets['train'])]
    train=spark.createDataFrame(rows,schema).repartition(2).cache()
    assert train.count()==72108
    results=[]
    try:
        for reg in (0.1,0.3):
            start=time.perf_counter()
            print('Training baseline reg',reg,flush=True)
            model=LogisticRegression(featuresCol='features',labelCol='label',family='multinomial',
                regParam=reg,elasticNetParam=0.0,maxIter=150,standardization=True,tol=1e-6).fit(train)
            coefficients=model.coefficientMatrix.toArray()
            intercept=model.interceptVector.toArray()
            probability=softmax(matrices['validation']@coefficients.T+intercept,axis=1)
            result=dict(model='Baseline LR',embedding='TF-IDF',reg_param=reg,iterations=int(model.summary.totalIterations),
                        hit_iteration_limit=int(model.summary.totalIterations)>=150,seconds=time.perf_counter()-start,
                        objective_history=list(model.summary.objectiveHistory),**evaluate(targets['validation'],probability))
            # Compare exported-parameter probabilities with Spark's own prediction path.
            check=matrices['validation'][:20]
            check_rows=[(float(i),Vectors.sparse(check.shape[1],check.indices[check.indptr[i]:check.indptr[i+1]],check.data[check.indptr[i]:check.indptr[i+1]])) for i in range(20)]
            transformed=model.transform(spark.createDataFrame(check_rows,schema)).select('label','probability').collect()
            for row in transformed:
                assert np.allclose(row.probability.toArray(),probability[int(row.label)],atol=1e-10,rtol=0)
            stem=f'lr_reg_{reg:g}'
            np.savez_compressed(OUT/f'{stem}_parameters.npz',coefficients=coefficients,intercept=intercept)
            np.savez_compressed(OUT/f'{stem}_validation.npz',target=targets['validation'],probability=probability)
            (OUT/f'{stem}.json').write_text(json.dumps(result,indent=2))
            results.append(result)
            (OUT/'summary.json').write_text(json.dumps(results,indent=2))
            print(stem,{k:result[k] for k in ('accuracy','macro_f1','precision_macro','recall_macro','iterations')},flush=True)
        best=max(results,key=lambda r:(r['macro_f1'],r['reg_param']))
        (OUT/'best.json').write_text(json.dumps(best,indent=2))
        print('Baseline completed and verified',flush=True)
    finally:
        spark.stop()

In [ ]:
# Run with the original split and saved TF-IDF artifacts in the repository root.
main()